## Database Connection (Config)

In [4]:
!pip install -q pymongo dnspython plotly pandas python-dotenv

import os
import json
from pymongo import MongoClient
from bson import json_util
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv('MONGO_URI')
DB_NAME = os.getenv('DB_NAME', 'tracera')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'raw_linkedin')

def connect_to_mongodb_and_get_data(uri, db_name, collection_name, query={}):
    if not uri:
        raise ValueError("Kredensial MONGO_URI tidak ditemukan! Pastikan file .env sudah ada.")
    try:
        client = MongoClient(uri)
        db = client[db_name]
        collection = db[collection_name]
        documents = list(collection.find(query))
        json_data = json.loads(json_util.dumps(documents))
        print(f"Berhasil terhubung! {len(json_data)} dokumen dimuat dari koleksi '{collection_name}'.")
        return json_data
    except Exception as e:
        print(f"Error koneksi: {e}")
        return []

raw_documents = connect_to_mongodb_and_get_data(URI, DB_NAME, COLLECTION_NAME)



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Berhasil terhubung! 397 dokumen dimuat dari koleksi 'raw_linkedin'.


## Pre-processing Data

In [5]:
from datetime import datetime, timezone
import pandas as pd

# ==============================================================================
# 1. KAMUS TRANSLASI & NORMALISASI SKILLS (Bilingual & Canonical Normalization)
# ==============================================================================
SKILL_TRANSLATIONS = {
    # a. Hard Skills - Desain, Media & Kreatif Teknis
    'desain': 'Design',
    'design': 'Design',
    'desain grafis': 'Graphic Design',
    'design grafis': 'Graphic Design',
    'graphic design': 'Graphic Design',
    'pemikiran desain': 'Design Thinking',
    'creative direction': 'Creative Direction',
    'creative strategy': 'Creative Strategy',
    'penulisan': 'Writing',
    'writing': 'Writing',
    'creative writing': 'Creative Writing',
    'penulisan kreatif': 'Creative Writing',
    'report writing': 'Report Writing',
    'penulisan laporan': 'Report Writing',
    'technical writing': 'Technical Writing',
    'penulisan teknis': 'Technical Writing',
    'scientific writing': 'Scientific Writing',
    'penulisan ilmiah': 'Scientific Writing',
    'proposal writing': 'Proposal Writing',
    'penulisan proposal': 'Proposal Writing',

    # b. Hard Skills - Data & Analisis
    'analisis data': 'Data Analysis',
    'analisa data': 'Data Analysis',
    'analisis': 'Data Analysis',
    'data analysis': 'Data Analysis',
    'analisis bisnis': 'Business Analysis',
    'business analysis': 'Business Analysis',
    'analisis keuangan': 'Financial Analysis',
    'financial analysis': 'Financial Analysis',
    'analisis sosial': 'Social Analysis',
    'pengolahan data': 'Data Processing',
    'entri data': 'Data Entry',
    'data entry': 'Data Entry',
    'mangalisis suatu proyek': 'Project Analysis',
    'riset': 'Research',
    'penelitian': 'Research',
    'research': 'Research',
    'research skills': 'Research',
    'analytical skills': 'Analytical Skills',
    'keahlian analitis': 'Analytical Skills',
    'analytic problem solving': 'Analytical Skills',

    # c. Hard Skills - Manajemen Proyek, Operasional & IT / Tools
    'manajemen proyek': 'Project Management',
    'project management': 'Project Management',
    'dokumen proyek': 'Project Documentation',
    'manajemen operasional': 'Operations Management',
    'operations management': 'Operations Management',
    'pengembangan web': 'Web Development',
    'manajemen file': 'File Management',
    'sistem informasi geografis (gis)': 'GIS',
    'geographic information systems (gis)': 'GIS',
    'arcgis': 'GIS',
    'sistem manajemen keselamatan dan kesehatan kerja': 'Occupational Health and Safety (OHS)',
    'kesehatan, keselamatan, dan lingkungan (hse)': 'Health, Safety, and Environment (HSE)',
    'peningkatan proses bisnis': 'Business Process Improvement',
    'monitoring dan evaluasi': 'Monitoring & Evaluation',
    'perencanaan strategis': 'Strategic Planning',
    'strategic planning': 'Strategic Planning',
    'manajemen perubahan': 'Change Management',
    'change management': 'Change Management',
    'organizational development': 'Organizational Development',
    'pengembangan organisasi': 'Organizational Development',
    'collaboration solutions': 'Collaboration Tools',
    'python (programming language)': 'Python',
    'go (programming language)': 'Go',
    'golang': 'Go',
    'c (programming language)': 'C',
    'r (programming language)': 'R',
    'java (programming language)': 'Java',
    'powerpoint': 'Microsoft PowerPoint',
    'excel': 'Microsoft Excel',
    'word': 'Microsoft Word',

    # d. Hard Skills - Keuangan, Akuntansi & Hukum Bisnis
    'keuangan': 'Finance',
    'finance': 'Finance',
    'jasa keuangan': 'Financial Services',
    'manajemen keuangan': 'Financial Management',
    'financial management': 'Financial Management',
    'perencanaan keuangan': 'Financial Planning',
    'financial planning': 'Financial Planning',
    'akuntansi': 'Accounting',
    'accounting': 'Accounting',
    'akuntansi keuangan': 'Financial Accounting',
    'financial accounting': 'Financial Accounting',
    'pembukuan': 'Bookkeeping',
    'bookkeeping': 'Bookkeeping',
    'perpajakan': 'Taxation',
    'tax': 'Taxation',
    'taxations': 'Taxation',
    'perencanaan anggaran': 'Budgeting',
    'budgeting': 'Budgeting',
    'budgets': 'Budgeting',
    'budget management': 'Budgeting',
    'auditing': 'Auditing',
    'internal audits': 'Auditing',
    'financial audits': 'Auditing',
    'financial reporting': 'Financial Reporting',
    'financial statements': 'Financial Reporting',
    'mengolah data biaya': 'Cost Accounting',
    'cost accounting': 'Cost Accounting',
    'konsultan manajemen': 'Management Consulting',
    'konsultan lingkungan': 'Environmental Consulting',
    'hukum': 'Legal & Law',
    'negosiasi kontrak': 'Contract Negotiation',
    'contract negotiation': 'Contract Negotiation',

    # e. Hard Skills - Pemasaran, Komunikasi Korporat, Penjualan & SDM
    'pemasaran': 'Marketing',
    'marketing': 'Marketing',
    'pemasaran digital': 'Digital Marketing',
    'pemasaran media sosial': 'Social Media Marketing',
    'social media marketing': 'Social Media Marketing',
    'pemasaran produk': 'Product Marketing',
    'strategi pemasaran': 'Marketing Strategy',
    'marketing strategy': 'Marketing Strategy',
    'strategi konten': 'Content Strategy',
    'content strategy': 'Content Strategy',
    'pengoptimalan media sosial (social media optimization/smo)': 'Social Media Optimization (SMO)',
    'komunikasi merek': 'Brand Communication',
    'brand communication': 'Brand Communication',
    'social media communications': 'Social Media Marketing',
    'hubungan masyarakat': 'Public Relations',
    'public relations': 'Public Relations',
    'penjualan': 'Sales',
    'sales': 'Sales',
    'perencanaan acara': 'Event Planning',
    'event planning': 'Event Planning',
    'event management': 'Event Planning',
    'sumber daya manusia (sdm)': 'Human Resources (HR)',
    'human resources (hr)': 'Human Resources (HR)',
    'human resources': 'Human Resources (HR)',
    'perencanaan sumber daya manusia strategis': 'Strategic Human Resource Planning',
    'administrasi': 'Administration',
    'administration': 'Administration',
    'office administration': 'Administration',
    'administrasi perkantoran': 'Administration',
    'bekerja di shift': 'Shift Work',
    'opname kerja': 'Stock Opname',

    # f. Hard Skills - Bahasa & Pengajaran
    'english': 'English',
    'bahasa inggris': 'English',
    'bahasa indonesia': 'Indonesian',
    'teaching': 'Teaching',
    'pengajaran': 'Teaching',

    # g. Soft Skills - Komunikasi & Interpersonal
    'komunikasi': 'Communication',
    'communication': 'Communication',
    'communications': 'Communication',
    'kemampuan komunikasi': 'Communication',
    'interpersonal communication': 'Communication',
    'written communication': 'Communication',
    'strategic communications': 'Communication',
    'intercultural communication': 'Communication',
    'cross-cultural communication skills': 'Communication',
    'komunikasi interpersonal': 'Communication',
    'komunikasi tertulis': 'Communication',
    'active listening': 'Active Listening',
    'mendengarkan aktif': 'Active Listening',
    'persuasi': 'Persuasion',
    'persuasion': 'Persuasion',
    'layanan pelanggan': 'Customer Service',
    'customer service': 'Customer Service',
    'pelayanan': 'Customer Service',
    'keahlian interpersonal': 'Interpersonal Skills',
    'interpersonal skills': 'Interpersonal Skills',
    'interpersonal': 'Interpersonal Skills',
    'keahlian presentasi': 'Presentation Skills',
    'presentation skills': 'Presentation Skills',
    'presentation': 'Presentation Skills',
    'presentations': 'Presentation Skills',
    'presentasi': 'Presentation Skills',
    'public speaking': 'Public Speaking',
    'berbicara di depan umum': 'Public Speaking',
    'relationship building': 'Relationship Building',
    'membangun relasi': 'Relationship Building',

    # h. Soft Skills - Tim & Kerjasama
    'kerja sama tim': 'Teamwork',
    'kerja tim': 'Teamwork',
    'kerjasama tim': 'Teamwork',
    'kemampuan kerja tim': 'Teamwork',
    'teamwork': 'Teamwork',
    'team work': 'Teamwork',
    'kolaborasi tim': 'Collaboration',
    'collaboration': 'Collaboration',
    'team collaboration': 'Collaboration',
    'cross-team collaboration': 'Collaboration',
    'cross-functional collaborations': 'Collaboration',
    'resolusi konflik': 'Conflict Resolution',
    'conflict resolution': 'Conflict Resolution',
    'conflict management': 'Conflict Resolution',
    'manajemen konflik': 'Conflict Resolution',

    # i. Soft Skills - Kepemimpinan & Manajemen Orang
    'kepemimpinan': 'Leadership',
    'leadership': 'Leadership',
    'team leadership': 'Leadership',
    'kepemimpinan tim': 'Leadership',
    'project leadership': 'Leadership',
    'organizational leadership': 'Leadership',
    'strategic leadership': 'Leadership',
    'cross-functional team leadership': 'Leadership',
    'leadership development': 'Leadership Development',
    'pengembangan kepemimpinan': 'Leadership Development',
    'manajemen tim': 'Team Management',
    'team management': 'Team Management',
    'team coordination': 'Team Management',
    'team building': 'Team Management',
    'team motivation': 'Team Management',
    'people management': 'People Management',
    'manajemen orang': 'People Management',
    'people development': 'People Development',
    'team development': 'People Development',
    'pengembangan tim': 'People Development',
    'coaching': 'Coaching & Mentoring',
    'mentoring': 'Coaching & Mentoring',
    'coaching & mentoring': 'Coaching & Mentoring',
    'training & mentoring': 'Coaching & Mentoring',
    'pelatihan & mentoring': 'Coaching & Mentoring',
    'sales coaching': 'Coaching & Mentoring',
    'kemampuan berorganisasi': 'Organization Skills',
    'organization skills': 'Organization Skills',
    'organizational skills': 'Organization Skills',
    'manajemen': 'Management',
    'management': 'Management',

    # j. Soft Skills - Pemecahan Masalah & Berpikir Kritis
    'pemecahan masalah': 'Problem Solving',
    'memecahkan masalah': 'Problem Solving',
    'problem solving': 'Problem Solving',
    'creative problem solving': 'Problem Solving',
    'collaborative problem solving': 'Problem Solving',
    'team problem solving': 'Problem Solving',
    'berpikir kritis': 'Critical Thinking',
    'critical thinking': 'Critical Thinking',
    'critical thinker': 'Critical Thinking',
    'analytical thinking': 'Critical Thinking',
    'pengambilan keputusan': 'Decision Making',
    'decision making': 'Decision Making',
    'decision-making': 'Decision Making',
    'data-driven decision making': 'Decision Making',
    'strategic thinking': 'Strategic Thinking',
    'creativity': 'Creativity',
    'kreativitas': 'Creativity',
    'creative thinking': 'Creativity',
    'creativity skills': 'Creativity',
    'creativity and innovation': 'Creativity',

    # k. Soft Skills - Manajemen Diri & Sikap Kerja
    'manajemen waktu': 'Time Management',
    'time management': 'Time Management',
    'manajemen diri': 'Self Management',
    'self management': 'Self Management',
    'adaptability': 'Adaptability',
    'adaptable': 'Adaptability',
    'easily adaptable': 'Adaptability',
    'adaptation': 'Adaptability',
    'flexibility': 'Adaptability',
    'negosiasi': 'Negotiation',
    'negotiation': 'Negotiation',
    'negotiation skills': 'Negotiation',
    'emotional intelligence': 'Emotional Intelligence',
    'work ethic': 'Work Ethic',
    'etika kerja': 'Work Ethic',
    'working under pressure': 'Working Under Pressure',
    'bekerja dalam tekanan': 'Working Under Pressure',
    'hard working': 'Work Ethic',
    'bekerja keras': 'Work Ethic',
    'jujur dan disiplin': 'Honesty & Discipline',
    'honesty & discipline': 'Honesty & Discipline',
    'bintal dan jasmani': 'Mental & Physical Discipline',
    'mental & physical discipline': 'Mental & Physical Discipline',
    'organizational culture': 'Organizational Culture'
}

# ==============================================================================
# 2. TAKSONOMI RESMI SOFT SKILLS (Bentuk Kanonikal Hasil Normalisasi)
# ==============================================================================
# Disimpan dalam huruf kecil untuk matching case-insensitive terhadap s_norm
SOFT_SKILLS = {
    # Komunikasi & Interpersonal
    'communication',
    'public speaking',
    'presentation skills',
    'active listening',
    'persuasion',
    'customer service',
    'interpersonal skills',
    'relationship building',

    # Tim, Kerjasama, & Hubungan
    'teamwork',
    'collaboration',
    'conflict resolution',

    # Kepemimpinan & People Management
    'leadership',
    'leadership development',
    'team management',
    'people management',
    'people development',
    'coaching & mentoring',
    'organization skills',
    'management',
    'organizational culture',

    # Pemecahan Masalah & Berpikir Kritis
    'problem solving',
    'critical thinking',
    'decision making',
    'strategic thinking',
    'creativity',

    # Adaptabilitas, Manajemen Waktu, & Sikap Kerja
    'adaptability',
    'time management',
    'self management',
    'negotiation',
    'emotional intelligence',
    'work ethic',
    'working under pressure',
    'honesty & discipline',
    'mental & physical discipline'
}

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def normalize_skill(name):
    """Membersihkan dan menstandardisasi nama keahlian."""
    if not name or not isinstance(name, str):
        return ''
    clean = name.strip()
    return SKILL_TRANSLATIONS.get(clean.lower(), clean)


def clean_job_title(title):
    """Membersihkan nama jabatan dari noise keterangan atau nama institusi."""
    if not title or not isinstance(title, str):
        return ''
    t = (
        title.split(' at ')[0]
        .split(' - ')[0]
        .split(' | ')[0]
        .split(' @ ')[0]
        .strip()
    )
    low = t.lower()
    noise_keywords = [
        'experience', 'endorsed', 'across', 'grade:', 'passed',
        'college', 'academy', 'universitas', 'gpa', 'alumni', "'--"
    ]
    if any(k in low for k in noise_keywords):
        return ''
    if len(t) < 3 or len(t) > 42:
        return ''
    return t

# ==============================================================================
# 4. FUNGSI PARSING DOKUMEN & PEMBERSIHAN ANOMALI
# ==============================================================================
def clean_doc(d):
    if not d or not isinstance(d, dict):
        return None

    raw_list = d.get('raw_data', [{}])
    p = raw_list[0] if isinstance(raw_list, list) and len(raw_list) > 0 else {}

    pid = p.get('publicIdentifier')
    if not pid or not isinstance(pid, str) or not pid.strip():
        return None

    # a. Validasi rentang tahun kerja & pembersihan jabatan
    experiences = []
    years = []
    for e in p.get('experience', []):
        yr = e.get('startDate', {}).get('year') if isinstance(e.get('startDate'), dict) else None
        if yr and str(yr).isdigit():
            yr_int = int(yr)
            if 1980 <= yr_int <= 2027:
                years.append(yr_int)
                clean_title = clean_job_title(e.get('position', ''))
                experiences.append({'year': yr_int, 'title': clean_title})

    # b. Industri (fallback ke 'Other')
    ind = p.get('industry')
    ind_name = ind.get('name') if isinstance(ind, dict) else (ind or 'Other')
    ind_clean = ind_name.strip() if isinstance(ind_name, str) and ind_name.strip() else 'Other'

    # c. Timestamp fetched_at
    fa = d.get('metadata', {}).get('fetched_at')
    try:
        fetched_dt = datetime.fromisoformat(fa) if fa else datetime.now(timezone.utc)
    except Exception:
        fetched_dt = datetime.now(timezone.utc)

    # d. Normalisasi Bahasa & Deduplikasi Skills per Profil
    skill_map = {}
    all_raw_s = [s['name'] for s in p.get('skills', []) if isinstance(s, dict) and s.get('name')]
    all_raw_s += (p.get('topSkills') or [])

    for s in all_raw_s:
        s_norm = normalize_skill(s)
        if len(s_norm) > 1:
            key = s_norm.lower()
            if key not in skill_map:
                skill_map[key] = s_norm
            elif s_norm[0].isupper() and not skill_map[key][0].isupper():
                skill_map[key] = s_norm

    # e. Program studi
    progs = []
    for e in p.get('education', []):
        fos = e.get('fieldOfStudy')
        if fos and isinstance(fos, str):
            fos_clean = fos.strip()
            if len(fos_clean) > 2 and fos_clean.lower() not in ['-', '.', '...', 'null', 'none']:
                if fos_clean not in progs:
                    progs.append(fos_clean)

    return {
        'id': pid.strip(),
        'industry': ind_clean,
        'fetched_at': fetched_dt,
        'skills': list(skill_map.values()),
        'programs': progs,
        'experiences': experiences,
        'latest_work_year': max(years, default=None)
    }

# ==============================================================================
# 5. BUAT DATAFRAME UTAMA & DEDUPLIKASI PROFIL
# ==============================================================================
raw_parsed = [clean_doc(d) for d in raw_documents if d]
raw_parsed = [doc for doc in raw_parsed if doc is not None]

df_raw = pd.DataFrame(raw_parsed)
df_profiles = df_raw.dropna(subset=['id']).drop_duplicates(subset=['id']).reset_index(drop=True)

df_skills = df_profiles.explode('skills').dropna(subset=['skills']).copy()
df_skills['skills_clean'] = df_skills['skills'].str.strip()

# Pengecekan tepat sasaran karena skills_clean sudah terstandardisasi
df_skills['is_soft'] = df_skills['skills_clean'].str.lower().isin(SOFT_SKILLS)

# ==============================================================================
# 6. RINGKASAN PREPROCESSING
# ==============================================================================
n_soft = int(df_skills['is_soft'].sum())
n_hard = int((~df_skills['is_soft']).sum())

print("=" * 55)
print("HASIL PREPROCESSING")
print("=" * 55)
print(f"- Data mentah masuk : {len(raw_documents)} dokumen")
print(f"- Total profil unik : {len(df_profiles)} alumni (duplikat dibersihkan)")
print(f"- Total baris skill : {len(df_skills)} baris")
print(f"- Total jenis skill : {df_skills['skills_clean'].nunique():,} skill unik")
print(f"  • Hard Skills     : {n_hard} baris ({round(n_hard/len(df_skills)*100, 1)}%)")
print(f"  • Soft Skills     : {n_soft} baris ({round(n_soft/len(df_skills)*100, 1)}%)")
print("=" * 55)

HASIL PREPROCESSING
- Data mentah masuk : 397 dokumen
- Total profil unik : 394 alumni (duplikat dibersihkan)
- Total baris skill : 5963 baris
- Total jenis skill : 2,658 skill unik
  • Hard Skills     : 5214 baris (87.4%)
  • Soft Skills     : 749 baris (12.6%)


## Dashboard Component 0: Key Performance Indicator

In [6]:
total_skill_unik = df_skills['skills_clean'].nunique()
rerata_skill_alumni = round(len(df_skills) / max(len(df_profiles), 1), 1)

alumni_terkini = df_profiles[df_profiles['latest_work_year'] >= 2026]
alumni_lama    = df_profiles[df_profiles['latest_work_year'] <= 2022]

keahlian_terkini = set(alumni_terkini.explode('skills')['skills'].dropna())
keahlian_lama    = set(alumni_lama.explode('skills')['skills'].dropna())

total_skill_tren = len(keahlian_terkini - keahlian_lama)

jumlah_skill_top = df_skills['skills_clean'].value_counts().iloc[0] if not df_skills.empty else 0
print("=" * 55)
print("KEY PERFORMANCE INDICATORS (KPI METRICS)")
print("=" * 55)
print(f"• Total Unique Skills        : {total_skill_unik:,} skill")
print(f"• Average Skills per Profile : {rerata_skill_alumni} keahlian / alumni")
print(f"• Trending Skills Count      : {total_skill_tren} keahlian era baru")
print(f"• Top Endorsed Skills        : {jumlah_skill_top} alumni")
print("=" * 55)

KEY PERFORMANCE INDICATORS (KPI METRICS)
• Total Unique Skills        : 2,658 skill
• Average Skills per Profile : 15.1 keahlian / alumni
• Trending Skills Count      : 1083 keahlian era baru
• Top Endorsed Skills        : 106 alumni


## Dashboard Component 1: Hard Skills vs Soft Skills (Bar Chart)

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

selected_industry = ""  # Contoh lain: "Mining", "Oil and Gas", atau "" (Semua)
top_n = 10

df_s_filt = df_skills[df_skills['industry'].str.lower() == selected_industry.lower()] if selected_industry else df_skills

top_hard = df_s_filt[~df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)
top_soft = df_s_filt[df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)


fig_a = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Top {top_n} Hard Skills", f"Top {top_n} Soft Skills"),
    shared_yaxes=False,
    horizontal_spacing=0.20
)

fig_a.add_trace(go.Bar(
    x=top_hard.values, y=top_hard.index,
    orientation='h', marker_color='#1d4ed8',
    name='Hard Skills', text=top_hard.values, textposition='auto'
), row=1, col=1)

fig_a.add_trace(go.Bar(
    x=top_soft.values, y=top_soft.index,
    orientation='h', marker_color='#10b981',
    name='Soft Skills', text=top_soft.values, textposition='auto'
), row=1, col=2)

fig_a.update_layout(
    title_text=f"<b>a. Pemetaan Hard Skills & Soft Skills Alumni</b> — Industri: {selected_industry or 'Semua'}",
    height=420,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_a.show()

## Dashboard Component 2: Skills Distribution by Category

## Dashboard Component 3: Network of Top Skills

Visualisasi jaringan interaktif untuk melihat korelasi dan keterhubungan antarkeahlian utama alumni.

- **Simpul (node)** merepresentasikan satu keterampilan; ukuran simpul menunjukkan jumlah alumni yang mencantumkannya.
- **Garis (edge)** menghubungkan dua keterampilan yang sering muncul bersama pada profil yang sama; ketebalan garis mengikuti kekuatan co-occurrence.
- **Keterampilan inti (core skills)** ditempatkan di pusat jaringan dan ditandai dengan cincin emas.
- **Interaksi:** arahkan kursor untuk melihat detail simpul/koneksi, scroll untuk zoom, dan klik-drag untuk menggeser kanvas.

In [8]:
from collections import Counter, defaultdict
from itertools import combinations
import numpy as np
import plotly.graph_objects as go

# ==============================================================================
# 0. CEK DEPENDENSI DATA
# ==============================================================================
# Cell ini membutuhkan hasil cell "Database Connection (Config)" dan
# "Pre-processing Data". Jalankan kedua cell tersebut (atau Runtime > Run all)
# sebelum menjalankan cell ini.
if 'df_profiles' not in globals():
    raise RuntimeError(
        "df_profiles belum tersedia. Jalankan cell 'Database Connection (Config)' "
        "dan 'Pre-processing Data' terlebih dahulu, lalu jalankan ulang cell ini "
        "(atau pilih Runtime > Run all)."
    )

if 'df_skills' not in globals():
    df_skills = (
        df_profiles[['id', 'industry', 'skills']]
        .explode('skills')
        .dropna(subset=['skills'])
        .rename(columns={'skills': 'skills_clean'})
        .reset_index(drop=True)
    )
    df_skills['skills_clean'] = df_skills['skills_clean'].astype(str).str.strip()

if 'SOFT_SKILLS' not in globals():
    if 'is_soft' in df_skills.columns:
        SOFT_SKILLS = set(
            df_skills.loc[df_skills['is_soft'].fillna(False), 'skills_clean'].str.lower()
        )
    else:
        raise RuntimeError(
            "SOFT_SKILLS belum tersedia. Jalankan cell 'Pre-processing Data' "
            "terlebih dahulu, lalu jalankan ulang cell ini."
        )

# ==============================================================================
# PARAMETER DASHBOARD
# ==============================================================================
top_n_skills       = 25   # Jumlah simpul (skill) yang ditampilkan
min_cooccurrence   = 4    # Minimal alumni pemilik kedua skill agar terbentuk garis
max_edges_per_node = 3    # Batasi garis terkuat per simpul agar graf tidak terlalu padat
core_count         = 3    # Jumlah keterampilan inti yang ditempatkan di pusat jaringan
selected_industry  = ''   # Contoh: "Mining", "Oil and Gas", atau "" (Semua)

# ==============================================================================
# 1. FILTER INDUSTRI & PEMILIHAN TOP SKILLS
# ==============================================================================
df_net_skills = df_skills.copy()
profiles = df_profiles.copy()

if selected_industry:
    df_net_skills = df_net_skills[
        df_net_skills['industry'].str.lower() == selected_industry.lower()
    ]
    profiles = profiles[
        profiles['industry'].str.lower() == selected_industry.lower()
    ]

skill_freq = df_net_skills['skills_clean'].value_counts()
top_skills = [s for s in skill_freq.head(top_n_skills).index.tolist() if isinstance(s, str)]
top_set = set(top_skills)

# ==============================================================================
# 2. MATRIKS CO-OCCURRENCE (KEHADIRAN BERSAMA DALAM SATU PROFIL)
# ==============================================================================
cooc = Counter()

for skills in profiles['skills']:
    if not isinstance(skills, (list, tuple, set)):
        continue
    unique_skills = sorted({s for s in skills if s in top_set})
    for skill_a, skill_b in combinations(unique_skills, 2):
        cooc[(skill_a, skill_b)] += 1

# ==============================================================================
# 3. PEMBENTUKAN EDGE: AMBANG BATAS + TOP-K PER SIMPUL
# ==============================================================================
candidates = {pair: weight for pair, weight in cooc.items() if weight >= min_cooccurrence}

neighbours = defaultdict(list)
for (skill_a, skill_b), weight in candidates.items():
    neighbours[skill_a].append((weight, skill_b))
    neighbours[skill_b].append((weight, skill_a))

edge_pairs = set()
for node, links in neighbours.items():
    for weight, other in sorted(links, key=lambda item: (-item[0], item[1]))[:max_edges_per_node]:
        edge_pairs.add(tuple(sorted((node, other))))

# Fallback: pastikan setiap simpul tetap memiliki minimal satu koneksi
for skill in top_skills:
    if any(skill in pair for pair in edge_pairs):
        continue

    strongest = max(
        [(cooc.get(tuple(sorted((skill, other))), 0), other)
         for other in top_skills if other != skill],
        default=(0, None)
    )
    if strongest[1] and strongest[0] > 0:
        edge_pairs.add(tuple(sorted((skill, strongest[1]))))

edges = [
    (skill_a, skill_b, cooc[(skill_a, skill_b)])
    for skill_a, skill_b in sorted(edge_pairs)
]

max_weight = max((weight for _, _, weight in edges), default=1)
n_profiles_net = max(len(profiles), 1)

# ==============================================================================
# 4. METRIK SIMPUL: DERAJAT, KEKUATAN, & KETERAMPILAN INTI
# ==============================================================================
degree = Counter()
strength = Counter()
edge_neighbours = defaultdict(list)

for skill_a, skill_b, weight in edges:
    degree[skill_a] += 1
    degree[skill_b] += 1
    strength[skill_a] += weight
    strength[skill_b] += weight
    edge_neighbours[skill_a].append((weight, skill_b))
    edge_neighbours[skill_b].append((weight, skill_a))

core_skills = [
    skill for skill, _ in sorted(
        strength.items(),
        key=lambda item: (-item[1], -skill_freq.get(item[0], 0), item[0])
    )[:core_count]
]
core_skills = core_skills or top_skills[:1]
core_set = set(core_skills)

# Setiap skill non-inti dimasukkan ke kluster keterampilan inti terdekat
cluster_of = {skill: skill for skill in core_set}
for skill in top_skills:
    if skill in core_set:
        continue
    scores = {
        core: cooc.get(tuple(sorted((skill, core))), 0)
        for core in core_skills
    }
    cluster_of[skill] = max(
        core_skills,
        key=lambda core: (scores[core], skill_freq.get(core, 0))
    )

# ==============================================================================
# 5. TATA LETAK RADIAL: INTI DI PUSAT, ANGGOTA MENGELILINGI KLUSTER
# ==============================================================================
nodes = top_skills
n_nodes = len(nodes)
node_index = {skill: i for i, skill in enumerate(nodes)}
positions = np.zeros((n_nodes, 2))

base_angle = np.pi / 2
core_angles = {
    core: base_angle - (2 * np.pi * i / max(len(core_skills), 1))
    for i, core in enumerate(core_skills)
}
core_radius = 0.62 if len(core_skills) > 1 else 0.0

for core in core_skills:
    positions[node_index[core]] = core_radius * np.array([
        np.cos(core_angles[core]),
        np.sin(core_angles[core])
    ])

sector_width = (2 * np.pi / max(len(core_skills), 1)) * 0.82

for core in core_skills:
    members = [skill for skill in nodes if skill not in core_set and cluster_of[skill] == core]
    members.sort(key=lambda skill: (
        -cooc.get(tuple(sorted((skill, core))), 0),
        -skill_freq.get(skill, 0),
        skill
    ))
    if not members:
        continue

    per_ring = max(4, int(np.ceil(len(members) / 2)))
    for rank, skill in enumerate(members):
        ring = rank // per_ring
        slot = rank % per_ring
        slot_count = min(per_ring, len(members) - ring * per_ring)
        fraction = 0.0 if slot_count == 1 else (slot - (slot_count - 1) / 2) / ((slot_count - 1) / 2)
        angle = core_angles[core] + fraction * sector_width / 2
        radius = 1.10 + 0.50 * ring
        positions[node_index[skill]] = radius * np.array([np.cos(angle), np.sin(angle)])

# Relaksasi pegas ringan: mengurangi tumpang tindih tanpa menghilangkan struktur kluster
core_indices = [node_index[core] for core in core_set]
weight_normalized = {
    (skill_a, skill_b): weight / max_weight
    for skill_a, skill_b, weight in edges
}

for iteration in range(120):
    displacement = np.zeros_like(positions)

    # Gaya tolak antar semua simpul
    for i in range(n_nodes):
        delta = positions[i] - positions
        distance = np.linalg.norm(delta, axis=1)
        distance[i] = np.inf
        distance = np.maximum(distance, 0.08)
        displacement[i] += np.sum((0.42 ** 2 / distance ** 2)[:, None] * delta, axis=0)

    # Gaya tarik pada setiap edge (semakin kuat co-occurrence, semakin dekat)
    for skill_a, skill_b, weight in edges:
        idx_a, idx_b = node_index[skill_a], node_index[skill_b]
        delta = positions[idx_a] - positions[idx_b]
        distance = max(np.linalg.norm(delta), 1e-6)
        force = (distance - 0.78) * (0.30 + 0.70 * weight_normalized[(skill_a, skill_b)])
        unit = delta / distance
        displacement[idx_a] -= unit * force
        displacement[idx_b] += unit * force

    # Gravitasi ringan agar graf tetap kompak, inti tetap di pusat
    displacement -= positions * 0.020
    displacement[core_indices] = 0

    temperature = 0.10 * (1 - iteration / 120) + 0.005
    lengths = np.linalg.norm(displacement, axis=1)
    scale = np.minimum(lengths, temperature) / np.maximum(lengths, 1e-9)
    positions += displacement * scale[:, None]

# Normalisasi memakai persentil 95 agar satu simpul outlier tidak mengecilkan graf
radius = np.linalg.norm(positions, axis=1)
extent = np.percentile(radius, 95)
if extent > 0:
    positions /= extent
    positions = np.clip(positions, -1.35, 1.35)

x_pos = positions[:, 0]
y_pos = positions[:, 1]

# ==============================================================================
# 6. PEWARNAAN & UKURAN SIMPUL
# ==============================================================================
max_freq = max(skill_freq.get(skill, 1) for skill in nodes)

skill_type = {
    skill: ('Soft Skills' if skill.lower() in SOFT_SKILLS else 'Hard Skills')
    for skill in nodes
}
node_sizes = {
    skill: 16 + 28 * np.sqrt(skill_freq.get(skill, 1) / max_freq) + (7 if skill in core_set else 0)
    for skill in nodes
}

CATEGORY_COLORS = {
    'Hard Skills': '#1d4ed8',
    'Soft Skills': '#10b981'
}


def label_position(x, y):
    """Menempatkan label di sisi luar simpul agar tidak menutupi garis."""
    if abs(x) >= abs(y):
        return 'middle right' if x >= 0 else 'middle left'
    return 'top center' if y >= 0 else 'bottom center'


# ==============================================================================
# 7. EDGE: WARNA PER JENIS PASANGAN & KETEBALAN MENGIKUTI KEKUATAN KONEKSI
# ==============================================================================
EDGE_COLORS = {
    'Hard – Hard': '#93c5fd',
    'Soft – Soft': '#6ee7b7',
    'Hard – Soft': '#cbd5e1',
}


def edge_layer(skill_a, skill_b):
    type_a, type_b = skill_type[skill_a], skill_type[skill_b]
    if type_a == type_b == 'Hard Skills':
        return 'Hard – Hard'
    if type_a == type_b == 'Soft Skills':
        return 'Soft – Soft'
    return 'Hard – Soft'


# Urutkan: per jenis pasangan, dari koneksi terlemah ke terkuat agar garis kuat berada di atas
edges_sorted = sorted(
    edges,
    key=lambda item: (list(EDGE_COLORS).index(edge_layer(item[0], item[1])), item[2])
)

fig_c = go.Figure()
legend_shown = set()

for skill_a, skill_b, weight in edges_sorted:
    layer_name = edge_layer(skill_a, skill_b)
    x_a, y_a = positions[node_index[skill_a]]
    x_b, y_b = positions[node_index[skill_b]]
    strength_ratio = np.sqrt(weight / max_weight)
    hover_text = (
        f"<b>{skill_a}</b> ↔ <b>{skill_b}</b><br>"
        f"{weight} alumni memiliki kedua skill ini<br>"
        f"Kekuatan koneksi: {weight / n_profiles_net * 100:.1f}% dari profil"
    )

    fig_c.add_trace(go.Scatter(
        x=[x_a, (x_a + x_b) / 2, x_b],
        y=[y_a, (y_a + y_b) / 2, y_b],
        mode='lines+markers',
        line=dict(
            color=EDGE_COLORS[layer_name],
            width=0.6 + 3.4 * strength_ratio,
            shape='spline',
        ),
        marker=dict(size=8, color='rgba(0,0,0,0)'),
        text=['', hover_text, ''],
        hovertemplate='%{text}<extra></extra>',
        name=layer_name,
        legendgroup=layer_name,
        showlegend=layer_name not in legend_shown,
        opacity=0.35 + 0.50 * strength_ratio,
    ))
    legend_shown.add(layer_name)

# ==============================================================================
# 8. SIMPUL PER KATEGORI + LABEL & HOVER
# ==============================================================================
for category in ['Hard Skills', 'Soft Skills']:
    category_nodes = [skill for skill in nodes if skill_type[skill] == category]
    if not category_nodes:
        continue

    customdata = []
    for skill in category_nodes:
        related = sorted(
            edge_neighbours.get(skill, []),
            key=lambda item: (-item[0], item[1])
        )[:3]
        related_text = ', '.join(
            f"{other} ({weight})" for weight, other in related
        ) or 'Belum ada koneksi kuat'
        customdata.append([
            skill,
            skill_type[skill],
            skill_freq.get(skill, 0),
            degree.get(skill, 0),
            strength.get(skill, 0),
            'Keterampilan Inti' if skill in core_set else cluster_of.get(skill, '-'),
            related_text,
        ])

    fig_c.add_trace(go.Scatter(
        x=[x_pos[node_index[skill]] for skill in category_nodes],
        y=[y_pos[node_index[skill]] for skill in category_nodes],
        mode='markers+text',
        name=category,
        text=category_nodes,
        textposition=[
            label_position(x_pos[node_index[skill]], y_pos[node_index[skill]])
            for skill in category_nodes
        ],
        textfont=dict(size=10, color='#0f172a', family='Arial'),
        marker=dict(
            size=[node_sizes[skill] for skill in category_nodes],
            color=CATEGORY_COLORS[category],
            line=dict(
                color=['#f59e0b' if skill in core_set else '#ffffff' for skill in category_nodes],
                width=[3.5 if skill in core_set else 1.5 for skill in category_nodes],
            ),
            opacity=0.95,
        ),
        customdata=customdata,
        cliponaxis=False,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Jenis: %{customdata[1]}<br>'
            'Jumlah alumni: %{customdata[2]}<br>'
            'Derajat koneksi: %{customdata[3]} skill<br>'
            'Kekuatan jaringan: %{customdata[4]}<br>'
            'Kluster inti: %{customdata[5]}<br>'
            '<i>Terhubung dengan:</i> %{customdata[6]}'
            '<extra></extra>'
        ),
    ))

# ==============================================================================
# 9. LAYOUT & RINGKASAN
# ==============================================================================
industry_label = selected_industry if selected_industry else 'Semua'
strongest_edge = max(edges, key=lambda item: item[2]) if edges else ('-', '-', 0)

fig_c.update_layout(
    title_text=(
        f'<b>c. Jaringan Keterhubungan Top Skills Alumni</b>'
        f' — Industri: {industry_label}'
    ),
    height=720,
    margin=dict(l=50, r=50, t=80, b=55),
    paper_bgcolor='white',
    plot_bgcolor='white',
    hovermode='closest',
    dragmode='pan',
    legend=dict(
        title=dict(text='<b>Keterangan</b>'),
        orientation='h',
        yanchor='bottom',
        y=1.01,
        xanchor='right',
        x=1,
    ),
    xaxis=dict(visible=False, scaleanchor='y', scaleratio=1, fixedrange=False),
    yaxis=dict(visible=False, fixedrange=False),
    annotations=[
        dict(
            text=(
                'Ukuran simpul = jumlah alumni · Ketebalan garis = jumlah alumni dengan kedua skill · '
                '<b>Cincin emas</b> = keterampilan inti (pusat jaringan)'
            ),
            xref='paper', yref='paper', x=0, y=-0.03,
            xanchor='left', yanchor='top',
            showarrow=False,
            font=dict(size=10, color='#64748b'),
        )
    ],
)

fig_c.show()

print("=" * 55)
print("NETWORK OF TOP SKILLS")
print("=" * 55)
print(f"- Simpul skill ditampilkan : {len(top_skills)}")
print(f"- Koneksi (co-occurrence)  : {len(edges)}")
print(f"- Keterampilan inti        : {', '.join(core_skills)}")
print(f"- Koneksi terkuat          : {strongest_edge[0]} ↔ {strongest_edge[1]} "
      f"({strongest_edge[2]} alumni)")
print("=" * 55)

NETWORK OF TOP SKILLS
- Simpul skill ditampilkan : 25
- Koneksi (co-occurrence)  : 65
- Keterampilan inti        : Communication, Leadership, Microsoft Office
- Koneksi terkuat          : Communication ↔ Leadership (55 alumni)


## Dashboard Component 4: Trending Skills

In [9]:
from collections import Counter, defaultdict
import pandas as pd
import plotly.graph_objects as go

# Parameter Konfigurasi Dashboard
top_n = 10
min_frequency = 10  # Ambang batas agar skill yang masuk benar-benar valid & berbobot
selected_industry = ''  # Contoh: "Mining", "Oil and Gas", atau "" (Semua)

# ==============================================================================
# 1. FILTER INDUSTRI BERDASARKAN DF_PROFILES
# ==============================================================================
profiles = df_profiles.copy()

if selected_industry:
    profiles = profiles[profiles['industry'].str.lower() == selected_industry.lower()]

# ==============================================================================
# 2. PEMBAGIAN BATCH TEMPORAL RELATIF (EARLIER 50% VS RECENT 50%)
# ==============================================================================
profiles_sorted = profiles.sort_values(by='fetched_at').reset_index(drop=True)

mid_point = len(profiles_sorted) // 2
earlier_profiles = profiles_sorted.iloc[:mid_point]
recent_profiles = profiles_sorted.iloc[mid_point:]

# ==============================================================================
# 3. PENGHITUNGAN FREKUENSI SKILL & LINKED ROLES
# ==============================================================================
skill_recent = Counter()
skill_earlier = Counter()
skill_linked_roles = defaultdict(Counter)

def aggregate_profile_data(df_subset, counter_target):
    for _, row in df_subset.iterrows():
        primary_role = (
            row['experiences'][0]['title'] 
            if row.get('experiences') and len(row['experiences']) > 0 
            else 'Specialist'
        )
        for s in row.get('skills', []):
            counter_target[s] += 1
            if primary_role:
                skill_linked_roles[s][primary_role] += 1

aggregate_profile_data(earlier_profiles, skill_earlier)
aggregate_profile_data(recent_profiles, skill_recent)

# ==============================================================================
# 4. KALKULASI & PENGURUTAN PERSENTASE TERBESAR KE TERKECIL
# ==============================================================================
all_skills = set(skill_earlier.keys()) | set(skill_recent.keys())
trending_list = []

for s_name in all_skills:
    c_rec = skill_recent[s_name]
    c_ear = skill_earlier[s_name]
    total_freq = c_rec + c_ear

    # Filter: frekuensi minimal 20 dan mengalami tren positif di batch recent
    if total_freq >= min_frequency and c_rec > c_ear:
        # Menghitung pangsa kemunculan pada window terbaru (0 - 100%)
        share_pct = (c_rec / total_freq) * 100

        top_roles = [r for r, _ in skill_linked_roles[s_name].most_common(2)]
        roles_text = ', '.join(top_roles) if top_roles else 'Professional Specialist'

        trending_list.append({
            'Skill Name': s_name,
            'Frequency': total_freq,
            'Share_Val': share_pct,
            'Recent Share (30d)': f'{share_pct:.1f}%',
            'Example linked roles': roles_text
        })

df_trending = pd.DataFrame(trending_list)

if not df_trending.empty:
    # Pengurutan langsung dari persentase terbesar ke terkecil
    df_trending = (
        df_trending.sort_values(by=['Share_Val', 'Frequency'], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )
else:
    df_trending = pd.DataFrame(columns=[
        'Skill Name', 'Frequency', 'Share_Val', 
        'Recent Share (30d)', 'Example linked roles'
    ])

# ==============================================================================
# 5. TABEL RINGKASAN INTERAKTIF PLOTLY
# ==============================================================================
industry_label = selected_industry if selected_industry else 'Semua'

cell_colors = (
    [['#f8fafc' if i % 2 == 0 else '#ffffff' for i in range(len(df_trending))]] * 4
    if len(df_trending) > 0
    else '#ffffff'
)

fig_d = go.Figure(
    data=[
        go.Table(
            columnwidth=[180, 90, 160, 310],
            header=dict(
                values=[
                    '<b>Skill Name</b>',
                    '<b>Total Frequency</b>',
                    '<b>Recent Share (30d)</b>',
                    '<b>Example linked roles</b>',
                ],
                fill_color='#0f172a',
                font=dict(color='white', size=12),
                align=['left', 'center', 'center', 'left'],
                height=36,
            ),
            cells=dict(
                values=[
                    df_trending['Skill Name'],
                    df_trending['Frequency'],
                    [f'<b>{val}</b>' for val in df_trending['Recent Share (30d)']],
                    df_trending['Example linked roles'],
                ],
                fill_color=cell_colors,
                font=dict(
                    color=['#0f172a', '#1e293b', '#16a34a', '#334155'], 
                    size=11
                ),
                align=['left', 'center', 'center', 'left'],
                height=34,
            ),
        )
    ]
)

fig_d.update_layout(
    title_text=(
        f'<b>d. Trending Skills: Pangsa Kemunculan 30 Hari Terakhir (0–100%)</b>'
        f' — Industri: {industry_label}'
    ),
    margin=dict(l=20, r=20, t=60, b=20),
    height=420,
)

fig_d.show()

## Dashboard Component 5: Role Readiness Gap

Mengukur kesiapan alumni untuk beralih ke role target berdasarkan skill yang dimiliki alumni yang sudah bekerja di role tersebut.

- **Role skill profile**: skill yang paling sering dimiliki alumni yang sudah bekerja di role target.
- **Readiness**: total bobot skill yang sudah dimiliki dibagi total bobot skill yang dibutuhkan, dalam persen.
- **Missing & prioritas**: skill yang belum dimiliki, diurutkan berdasarkan prevalensi skill di role target × besarnya gap kepemilikan.
- **Mode analisis**: kosongkan `selected_profile_id` untuk melihat agregat alumni di luar role target, atau isi ID alumni untuk analisis individu.


In [10]:
from collections import Counter
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==============================================================================
# 0. CEK DEPENDENSI DATA
# ==============================================================================
if 'df_profiles' not in globals():
    raise RuntimeError(
        "df_profiles belum tersedia. Jalankan cell 'Database Connection (Config)' "
        "dan 'Pre-processing Data' terlebih dahulu, lalu jalankan ulang cell ini "
        "(atau pilih Runtime > Run all)."
    )

# ==============================================================================
# PARAMETER DASHBOARD
# ==============================================================================
target_role = 'Data Analyst'   # Contoh lain: 'Data Scientist', 'Finance Staff', 'HR Specialist'
selected_profile_id = ''        # Kosongkan untuk mode agregat; isi ID alumni untuk mode individu
min_role_holders = 2            # Minimal alumni role target agar profil skill dianggap valid
min_skill_share = 0.30          # Minimal proporsi alumni role target yang memiliki skill
max_role_skills = 10            # Maksimal skill yang masuk profil role

# ==============================================================================
# 1. IDENTIFIKASI ALUMNI YANG SUDAH BEKERJA DI ROLE TARGET
# ==============================================================================
df_roles = df_profiles.copy()


def get_latest_title(experiences):
    """Mengambil jabatan dari pengalaman kerja dengan tahun terbaru."""
    if not isinstance(experiences, list) or not experiences:
        return ''
    valid = [e for e in experiences if isinstance(e, dict) and e.get('title')]
    if not valid:
        return ''
    latest = max(valid, key=lambda e: e.get('year') or 0)
    return str(latest.get('title', '')).strip()


def title_has(title, keyword):
    return isinstance(title, str) and keyword in title.lower()


def experience_has(experiences, keyword):
    if not isinstance(experiences, list):
        return False
    return any(
        title_has(e.get('title', ''), keyword)
        for e in experiences if isinstance(e, dict)
    )


target_keyword = target_role.strip().lower()
if not target_keyword:
    raise ValueError("target_role tidak boleh kosong.")

df_roles['latest_title'] = df_roles['experiences'].apply(get_latest_title)

role_mask = df_roles['latest_title'].apply(lambda title: title_has(title, target_keyword))
match_note = 'jabatan terbaru'

if role_mask.sum() < min_role_holders:
    role_mask = df_roles['experiences'].apply(
        lambda experiences: experience_has(experiences, target_keyword)
    )
    match_note = 'riwayat jabatan'

# Fallback: cocokkan per kata kunci (mis. "Data Analyst" -> "Data Scientist"/"Business Analyst")
if role_mask.sum() < min_role_holders:
    tokens = [t for t in re.split(r'[^a-z0-9]+', target_keyword) if len(t) >= 3]

    if tokens:
        role_mask = df_roles['latest_title'].apply(
            lambda title: any(tok in str(title).lower() for tok in tokens)
        )
        match_note = 'jabatan terbaru (pencocokan diperluas)'

    if role_mask.sum() < min_role_holders:
        role_mask = df_roles['experiences'].apply(
            lambda experiences: any(
                any(tok in str(e.get('title', '')).lower() for tok in tokens)
                for e in experiences if isinstance(e, dict)
            )
        )
        match_note = 'riwayat jabatan (pencocokan diperluas)'

role_holders = df_roles[role_mask].copy()
n_role_holders = len(role_holders)

if n_role_holders == 0:
    available = (
        df_roles['latest_title']
        .replace('', np.nan)
        .dropna()
        .value_counts()
        .head(15)
    )
    daftar_role = ', '.join(available.index.tolist()) or '(tidak ada data jabatan)'
    raise ValueError(
        f"Tidak ada alumni dengan role '{target_role}'. "
        f"Contoh role yang tersedia: {daftar_role}. "
        "Ubah nilai target_role lalu jalankan ulang cell ini."
    )

# ==============================================================================
# 2. MEMBANGUN ROLE SKILL PROFILE
# ==============================================================================
role_skill_counts = Counter()
for skills in role_holders['skills']:
    if not isinstance(skills, (list, tuple, set)):
        continue
    for skill in set(skills):
        role_skill_counts[skill] += 1

if not role_skill_counts:
    raise ValueError(
        f"Alumni dengan role '{target_role}' belum memiliki data skill yang cukup."
    )

skill_weights = pd.Series(role_skill_counts, dtype=float) / n_role_holders
skill_weights = skill_weights.sort_values(ascending=False)

required_skills = skill_weights[skill_weights >= min_skill_share].head(max_role_skills)
if len(required_skills) < 5:
    required_skills = skill_weights.head(max_role_skills)

required_skills = required_skills.sort_values(ascending=False)
required_list = required_skills.index.tolist()
weights = required_skills.values
weight_sum = weights.sum()

if len(required_list) == 0 or weight_sum <= 0:
    raise ValueError("Profil skill role target kosong. Periksa kembali data alumni.")

# ==============================================================================
# 3. TENTUKAN ALUMNI YANG DINILAI (INDIVIDU ATAU AGREGAT)
# ==============================================================================
if str(selected_profile_id).strip():
    selected_profile_id = str(selected_profile_id).strip()
    selected_rows = df_profiles[df_profiles['id'] == selected_profile_id]

    if selected_rows.empty:
        contoh_id = ', '.join(df_profiles['id'].head(5).tolist())
        raise ValueError(
            f"ID alumni '{selected_profile_id}' tidak ditemukan. "
            f"Contoh ID yang tersedia: {contoh_id}"
        )

    assessed = selected_rows.iloc[[0]]
    assessed_label = f"Alumni: {selected_profile_id}"
else:
    assessed = df_roles[~role_mask].copy()
    if len(assessed) < 2:
        assessed = df_roles.copy()
    assessed_label = f"Agregat {len(assessed)} alumni di luar role target"

assessed_skill_sets = [
    set(skills) for skills in assessed['skills']
    if isinstance(skills, (list, tuple, set))
]

if not assessed_skill_sets:
    raise ValueError("Alumni yang dinilai belum memiliki data skill.")


def coverage_of(skill_set):
    """Proporsi bobot skill role yang sudah dimiliki."""
    return sum(
        weight for skill, weight in zip(required_list, weights)
        if skill in skill_set
    ) / weight_sum


# ==============================================================================
# 4. HITUNG READINESS, MISSING SKILL, & PRIORITAS
# ==============================================================================
if str(selected_profile_id).strip():
    target_skills = assessed_skill_sets[0]
    ownership = np.array([
        1.0 if skill in target_skills else 0.0
        for skill in required_list
    ])
    readiness = coverage_of(target_skills) * 100
else:
    ownership = np.array([
        sum(1 for skill_set in assessed_skill_sets if skill in skill_set)
        / len(assessed_skill_sets)
        for skill in required_list
    ])
    readiness = np.mean([coverage_of(skill_set) for skill_set in assessed_skill_sets]) * 100

gap_df = pd.DataFrame({
    'Skill': required_list,
    'Prevalensi': weights,
    'Kepemilikan': ownership,
    'Gap': 1 - ownership,
})
gap_df['Skor Prioritas'] = gap_df['Prevalensi'] * gap_df['Gap']
gap_df['Status'] = np.where(
    gap_df['Kepemilikan'] >= 0.5,
    'Sudah dimiliki',
    'Masih kurang'
)
gap_df = (
    gap_df.sort_values(['Skor Prioritas', 'Prevalensi'], ascending=[False, False])
    .reset_index(drop=True)
)
gap_df['Prioritas'] = np.arange(1, len(gap_df) + 1)

owned_skills = gap_df.loc[gap_df['Status'] == 'Sudah dimiliki', 'Skill'].tolist()
missing_skills = gap_df.loc[gap_df['Status'] == 'Masih kurang', 'Skill'].tolist()


def readiness_color(value):
    if value >= 75:
        return '#16a34a'
    if value >= 50:
        return '#f59e0b'
    return '#dc2626'


# ==============================================================================
# 5. VISUALISASI: GAUGE + STACKED BAR + TABEL PRIORITAS
# ==============================================================================
plot_df = gap_df.sort_values('Skor Prioritas', ascending=True).reset_index(drop=True)
plot_skills = plot_df['Skill'].tolist()
own_pct = (plot_df['Kepemilikan'] * 100).tolist()
gap_pct = (plot_df['Gap'] * 100).tolist()
own_colors = [
    '#10b981' if status == 'Sudah dimiliki' else '#ef4444'
    for status in plot_df['Status']
]

fig_e = make_subplots(
    rows=2, cols=2,
    specs=[
        [{'type': 'indicator'}, {'type': 'xy'}],
        [{'type': 'table', 'colspan': 2}, None],
    ],
    row_heights=[0.40, 0.60],
    column_widths=[0.30, 0.70],
    vertical_spacing=0.10,
    subplot_titles=(
        'Readiness',
        f'Kepemilikan Skill vs Kebutuhan Role: {target_role}',
        'Rincian Skill Gap & Prioritas',
    ),
)

fig_e.add_trace(
    go.Indicator(
        mode='gauge+number',
        value=readiness,
        number={'suffix': '%', 'font': {'size': 40, 'color': '#0f172a'}},
        gauge={
            'axis': {
                'range': [0, 100],
                'tickvals': [0, 25, 50, 75, 100],
                'tickfont': {'size': 10, 'color': '#64748b'},
            },
            'bar': {'color': readiness_color(readiness), 'thickness': 0.30},
            'bgcolor': '#f8fafc',
            'borderwidth': 0,
            'steps': [
                {'range': [0, 50], 'color': '#fee2e2'},
                {'range': [50, 75], 'color': '#fef3c7'},
                {'range': [75, 100], 'color': '#dcfce7'},
            ],
        },
    ),
    row=1, col=1,
)

fig_e.add_trace(
    go.Bar(
        y=plot_skills,
        x=own_pct,
        orientation='h',
        name='Sudah dimiliki',
        marker=dict(color=own_colors, line=dict(color='white', width=1)),
        text=[f'{value:.0f}%' if value >= 8 else '' for value in own_pct],
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Sudah dimiliki: %{x:.1f}% alumni<extra></extra>'
        ),
    ),
    row=1, col=2,
)

fig_e.add_trace(
    go.Bar(
        y=plot_skills,
        x=gap_pct,
        orientation='h',
        name='Gap',
        marker=dict(color='#fecaca', line=dict(color='white', width=1)),
        text=[f'{value:.0f}%' if value >= 8 else '' for value in gap_pct],
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Gap: %{x:.1f}% alumni<extra></extra>'
        ),
    ),
    row=1, col=2,
)

table_df = gap_df.copy()
row_fills = [
    '#f0fdf4' if status == 'Sudah dimiliki' else '#fef2f2'
    for status in table_df['Status']
]

fig_e.add_trace(
    go.Table(
        columnwidth=[55, 190, 120, 105, 110, 80, 115],
        header=dict(
            values=[
                '<b>Prioritas</b>',
                '<b>Skill</b>',
                '<b>Status</b>',
                '<b>Prevalensi Role</b>',
                '<b>Kepemilikan</b>',
                '<b>Gap</b>',
                '<b>Skor Prioritas (0–100)</b>',
            ],
            fill_color='#0f172a',
            font=dict(color='white', size=11),
            align=['center', 'left', 'left', 'center', 'center', 'center', 'center'],
            height=34,
        ),
        cells=dict(
            values=[
                table_df['Prioritas'],
                table_df['Skill'],
                table_df['Status'],
                [f'{value * 100:.0f}%' for value in table_df['Prevalensi']],
                [f'{value * 100:.0f}%' for value in table_df['Kepemilikan']],
                [f'{value * 100:.0f}%' for value in table_df['Gap']],
                [f'{value * 100:.1f}' for value in table_df['Skor Prioritas']],
            ],
            fill_color=[row_fills] * 7,
            font=dict(color='#0f172a', size=11),
            align=['center', 'left', 'left', 'center', 'center', 'center', 'center'],
            height=30,
        ),
    ),
    row=2, col=1,
)

fig_e.update_layout(
    title_text=(
        f'<b>e. Role Readiness Gap: {target_role}</b>'
        f' — {assessed_label}'
    ),
    title_x=0.01,
    height=820,
    margin=dict(l=30, r=30, t=90, b=30),
    barmode='stack',
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial', color='#0f172a'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.01,
        xanchor='right',
        x=1,
    ),
)

fig_e.update_xaxes(
    range=[0, 100],
    title_text='Persentase alumni',
    gridcolor='#e2e8f0',
    row=1, col=2,
)
fig_e.update_yaxes(title_text='', row=1, col=2)
fig_e.update_annotations(font=dict(size=12, color='#334155'))

fig_e.show()

# ==============================================================================
# 6. RINGKASAN TEKS
# ==============================================================================
if str(selected_profile_id).strip():
    top_priorities = ', '.join(
        row['Skill'] for _, row in gap_df.head(3).iterrows()
    )
else:
    top_priorities = ', '.join(
        f"{row['Skill']} ({row['Kepemilikan'] * 100:.0f}% dimiliki)"
        for _, row in gap_df.head(3).iterrows()
    )

print("=" * 55)
print("ROLE READINESS GAP")
print("=" * 55)
print(f"Target role       : {target_role}")
print(f"Role holders      : {n_role_holders} alumni ({match_note})")
print(f"Alumni dinilai    : {assessed_label}")
print(f"Readiness         : {readiness:.1f}%")
print(f"Sudah dimiliki    : {', '.join(owned_skills) if owned_skills else '-'}")
print(f"Masih kurang      : {', '.join(missing_skills) if missing_skills else '-'}")
print(f"Prioritas utama   : {top_priorities}")
print("=" * 55)

ROLE READINESS GAP
Target role       : Data Analyst
Role holders      : 3 alumni (riwayat jabatan)
Alumni dinilai    : Agregat 391 alumni di luar role target
Readiness         : 9.7%
Sudah dimiliki    : -
Masih kurang      : Python, Finance, Analytical Skills, Microsoft Excel, Communication, Cloudera Machine Learning, Cloudera, RStudio, Social Media Marketing, Adobe Photoshop
Prioritas utama   : Python (4% dimiliki), Finance (5% dimiliki), Analytical Skills (15% dimiliki)


## Dashboard Component 6: Skills Intelligence Map (Interaktif)

Satu diagram utama yang bisa diklik. Setiap komponen di dalamnya langsung
dijelaskan lewat panel kecil berisi mini-chart dan deskripsi, sehingga kesimpulan
tidak perlu dibaca dari banyak grafik terpisah.

**Cara pakai**

- **Klik gelembung skill** - panel kanan menampilkan detail skill: tipe, kuadran,
  persentase alumni, momentum, skor keterhubungan, prioritas di role target,
  mini-chart kepemilikan vs kebutuhan vs tren, dan rekomendasi tindakan.
- **Klik label kuadran** (Naik Daun, Kekuatan Inti, Niche, Kompetensi Mapan) -
  panel menampilkan penjelasan kuadran, statistik ringkas, dan skill terbesarnya.
- **Klik area kosong atau tombol Ringkasan** - kembali ke kesimpulan umum.
- **Arahkan kursor** ke gelembung - muncul tooltip ringkas tanpa mengubah panel.

**Komponen yang disintesis**

| Elemen diagram | Sumber | Arti |
| --- | --- | --- |
| Sumbu X (Supply) | Komponen 0 & 1 | Persentase alumni yang memiliki skill |
| Sumbu Y (Momentum) | Komponen 4 | Perubahan pangsa alumni antara data lama dan terbaru |
| Ukuran gelembung | Komponen 0 | Jumlah alumni pemilik skill |
| Warna | Komponen 1 | Biru = hard skill, hijau = soft skill |
| Cincin emas | Komponen 5 | Skill yang diminta role target |
| Skor keterhubungan | Komponen 3 | Kekuatan co-occurrence dengan skill lain |

**Kuadran**

- **Naik Daun** - langka tetapi menguat, kandidat kurikulum baru.
- **Kekuatan Inti** - melimpah dan menguat, jadi pembeda utama alumni.
- **Niche** - langka dan melemah, cukup dipantau.
- **Kompetensi Mapan** - melimpah tetapi melemah, perlu dijaga relevansinya.

**Output**

1. Widget interaktif yang tampil langsung di notebook (HTML/SVG/JavaScript,
   tanpa dependensi tambahan).
2. File `skills_intelligence.html` yang mandiri dan bisa dibuka di browser atau
   di-embed ke halaman Skills Intelligence di web Tracera.

Ubah parameter `target_role`, `selected_industry`, dan `top_n_skills` untuk
menyesuaikan kesimpulan dengan konteks yang sedang dianalisis.

In [11]:
from collections import Counter, defaultdict
from itertools import combinations
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

# ==============================================================================
# 0. CEK DEPENDENSI DATA
# ==============================================================================
if 'df_profiles' not in globals():
    raise RuntimeError(
        "df_profiles belum tersedia. Jalankan cell 'Database Connection (Config)' "
        "dan 'Pre-processing Data' terlebih dahulu, lalu jalankan ulang cell ini "
        "(atau pilih Runtime > Run all)."
    )

if 'df_skills' not in globals():
    df_skills = (
        df_profiles[['id', 'industry', 'skills']]
        .explode('skills')
        .dropna(subset=['skills'])
        .rename(columns={'skills': 'skills_clean'})
        .reset_index(drop=True)
    )
    df_skills['skills_clean'] = df_skills['skills_clean'].astype(str).str.strip()

if 'SOFT_SKILLS' not in globals():
    if 'is_soft' in df_skills.columns:
        SOFT_SKILLS = set(
            df_skills.loc[df_skills['is_soft'].fillna(False), 'skills_clean'].str.lower()
        )
    else:
        SOFT_SKILLS = set()
        print(
            "Peringatan: daftar SOFT_SKILLS tidak tersedia, sehingga seluruh skill "
            "dihitung sebagai hard skill. Jalankan cell 'Pre-processing Data' agar "
            "komposisi hard vs soft akurat."
        )

# ==============================================================================
# PARAMETER DASHBOARD
# ==============================================================================
target_role       = 'Data Analyst'  # Role acuan kesimpulan
selected_industry = ''              # Contoh: 'Mining', 'Oil and Gas', atau '' (Semua)
top_n_skills      = 18              # Jumlah gelembung skill di diagram utama
min_skill_count   = 5               # Minimal alumni pemilik skill
min_cooccurrence  = 4               # Minimal alumni pemilik dua skill agar dihitung terhubung
role_min_share    = 0.30            # Minimal prevalensi skill agar masuk profil role
role_max_skills   = 10              # Maksimal skill dalam profil role
export_html_path  = 'skills_intelligence.html'  # File HTML mandiri untuk web

# ==============================================================================
# 1. FILTER DATA & INDIKATOR DASAR (KOMPONEN 0)
# ==============================================================================
df_f = df_skills.copy()
profiles = df_profiles.copy()

if selected_industry:
    df_f = df_f[df_f['industry'].str.lower() == selected_industry.lower()]
    profiles = profiles[profiles['industry'].str.lower() == selected_industry.lower()]

df_f = df_f.copy()
if 'is_soft' not in df_f.columns:
    df_f['is_soft'] = df_f['skills_clean'].str.lower().isin(SOFT_SKILLS)
df_f['is_soft'] = df_f['is_soft'].fillna(False)

n_alumni = max(profiles['id'].nunique(), 1)
n_skill_rows = len(df_f)
n_unique_skills = df_f['skills_clean'].nunique()
n_soft_rows = int(df_f['is_soft'].sum())
n_hard_rows = n_skill_rows - n_soft_rows
soft_pct = n_soft_rows / n_skill_rows * 100 if n_skill_rows else 0.0
hard_pct = 100 - soft_pct
avg_skills = n_skill_rows / n_alumni
industry_label = selected_industry if selected_industry else 'Semua'

# ==============================================================================
# 2. SUPPLY: PANGSA ALUMNI YANG MEMILIKI SKILL (KOMPONEN 0 & 1)
# ==============================================================================
skill_counts = df_f['skills_clean'].value_counts()
skill_counts = skill_counts[skill_counts >= min_skill_count]

if skill_counts.empty:
    # Dataset kecil: longgarkan ambang agar diagram tetap bisa ditampilkan
    skill_counts = df_f['skills_clean'].value_counts().head(top_n_skills)
if skill_counts.empty:
    raise ValueError(
        "Tidak ada data skill pada filter yang dipilih. Ubah selected_industry "
        "atau pastikan cell 'Pre-processing Data' sudah dijalankan."
    )

supply_pct = skill_counts / n_alumni * 100

# ==============================================================================
# 3. MOMENTUM: BATCH TERBARU VS TERLAMA (KOMPONEN 4)
# ==============================================================================
profiles = profiles.copy()
profiles['_when'] = pd.to_datetime(profiles['fetched_at'], errors='coerce')
if profiles['_when'].isna().all():
    profiles['_when'] = pd.Timestamp('2025-01-01')
else:
    profiles['_when'] = profiles['_when'].fillna(profiles['_when'].min())

profiles_sorted = profiles.sort_values('_when').reset_index(drop=True)
mid_point = len(profiles_sorted) // 2
earlier = profiles_sorted.iloc[:mid_point]
recent = profiles_sorted.iloc[mid_point:]


def count_unique_skills(df_subset):
    counter = Counter()
    for skills in df_subset['skills']:
        if isinstance(skills, (list, tuple, set)):
            for skill in set(skills):
                if isinstance(skill, str):
                    counter[skill] += 1
    return counter


count_earlier = count_unique_skills(earlier)
count_recent = count_unique_skills(recent)
n_earlier = max(len(earlier), 1)
n_recent = max(len(recent), 1)


def share_of(counter, total, skill):
    return counter.get(skill, 0) / total * 100


def momentum_of(skill):
    return share_of(count_recent, n_recent, skill) - share_of(count_earlier, n_earlier, skill)


# ==============================================================================
# 4. PROFIL ROLE TARGET & READINESS (KOMPONEN 5)
# ==============================================================================
def latest_title(experiences):
    if not isinstance(experiences, list) or not experiences:
        return ''
    valid = [e for e in experiences if isinstance(e, dict) and e.get('title')]
    if not valid:
        return ''
    latest = max(valid, key=lambda e: e.get('year') or 0)
    return str(latest.get('title', '')).strip()


def experiences_match(experiences, keyword):
    if not isinstance(experiences, list):
        return False
    return any(
        isinstance(e, dict) and keyword in str(e.get('title', '')).lower()
        for e in experiences
    )


role_keyword = target_role.strip().lower()
role_titles = profiles['experiences'].apply(latest_title)
role_mask = role_titles.str.lower().str.contains(role_keyword, regex=False)
role_note = 'jabatan terbaru'

if role_mask.sum() < 2 and role_keyword:
    role_mask = profiles['experiences'].apply(
        lambda exps: experiences_match(exps, role_keyword)
    )
    role_note = 'riwayat jabatan'

if role_mask.sum() < 2 and role_keyword:
    tokens = [t for t in role_keyword.replace(',', ' ').split() if len(t) >= 3]
    if tokens:
        role_mask = role_titles.apply(
            lambda title: any(tok in str(title).lower() for tok in tokens)
        )
        role_note = 'jabatan terbaru (pencocokan diperluas)'

role_holders = profiles[role_mask]
role_ready = len(role_holders) > 0

role_required = pd.Series(dtype=float)
role_prev = pd.Series(dtype=float)
c6_gap = pd.DataFrame(columns=['Skill', 'Prevalensi', 'Kepemilikan', 'Gap', 'Skor Prioritas'])
c6_readiness = 0.0

if role_ready:
    role_counts = Counter()
    for skills in role_holders['skills']:
        if isinstance(skills, (list, tuple, set)):
            for skill in set(skills):
                role_counts[skill] += 1

    role_prev = (pd.Series(role_counts, dtype=float) / len(role_holders)).sort_values(
        ascending=False
    )
    role_required = role_prev[role_prev >= role_min_share].head(role_max_skills)
    if len(role_required) < 5:
        role_required = role_prev.head(role_max_skills)
    role_required = role_required.sort_values(ascending=False)

    assessed = profiles[~role_mask]
    if len(assessed) < 2:
        assessed = profiles.copy()

    weight_sum = role_required.sum()
    coverage = []
    for skills in assessed['skills']:
        skill_set = set(skills) if isinstance(skills, (list, tuple, set)) else set()
        coverage.append(
            sum(weight for skill, weight in role_required.items() if skill in skill_set)
            / weight_sum
        )
    c6_readiness = float(np.mean(coverage) * 100) if coverage else 0.0

    ownership = np.array([
        sum(1 for skills in assessed['skills']
            if isinstance(skills, (list, tuple, set)) and skill in skills)
        / max(len(assessed), 1)
        for skill in role_required.index
    ])

    c6_gap = pd.DataFrame({
        'Skill': role_required.index,
        'Prevalensi': role_required.values,
        'Kepemilikan': ownership,
        'Gap': 1 - ownership,
    })
    c6_gap['Skor Prioritas'] = c6_gap['Prevalensi'] * c6_gap['Gap'] * 100
    c6_gap = c6_gap.sort_values('Skor Prioritas', ascending=False).reset_index(drop=True)

role_skill_set = set(role_required.index) if role_ready else set()
role_rank = {
    row['Skill']: index + 1
    for index, (_, row) in enumerate(c6_gap.iterrows())
} if role_ready else {}

# ==============================================================================
# 5. KETERHUBUNGAN SKILL (KOMPONEN 3)
# ==============================================================================
selected_skills = skill_counts.head(top_n_skills).index.tolist()
for skill in c6_gap['Skill'].head(5):
    if skill in skill_counts.index and skill not in selected_skills:
        selected_skills.append(skill)

top_set = set(selected_skills)
cooc = Counter()
for skills in profiles['skills']:
    if not isinstance(skills, (list, tuple, set)):
        continue
    present = sorted({s for s in skills if s in top_set})
    for skill_a, skill_b in combinations(present, 2):
        cooc[(skill_a, skill_b)] += 1

partners = Counter()
network_strength = Counter()
for (skill_a, skill_b), weight in cooc.items():
    if weight >= min_cooccurrence:
        partners[skill_a] += 1
        partners[skill_b] += 1
        network_strength[skill_a] += weight
        network_strength[skill_b] += weight

# ==============================================================================
# 6. KUADRAN & POSISI LABEL
# ==============================================================================
hero_skills = [s for s in selected_skills if s in supply_pct.index]
supply_values = np.array([supply_pct[s] for s in hero_skills])
momentum_values = np.array([momentum_of(s) for s in hero_skills])
alumni_values = np.array([skill_counts[s] for s in hero_skills])

x_max = max(supply_values.max() * 1.14, 5.0)
y_span = max(momentum_values.max() - momentum_values.min(), 4.0)
y_min = momentum_values.min() - y_span * 0.12
y_max = momentum_values.max() + y_span * 0.12
x_mid = float(np.median(supply_values))


def quadrant_of(supply, momentum):
    if momentum >= 0:
        return 'naik' if supply <= x_mid else 'inti'
    return 'niche' if supply <= x_mid else 'mapan'


QUADRANTS = {
    'naik': {
        'label': 'Naik Daun',
        'short': 'NAIK DAUN',
        'desc': (
            'Skill langka yang justru makin sering muncul pada profil terbaru. '
            'Kuadran ini adalah sinyal kebutuhan masa depan: paling layak menjadi '
            'modul pelatihan baru dan pembeda lulusan.'
        ),
        'action': 'Rekomendasi: masukkan ke kurikulum sebagai skill prioritas.',
        'color': '#059669',
        'fill': '#ecfdf5',
    },
    'inti': {
        'label': 'Kekuatan Inti',
        'short': 'KEKUATAN INTI',
        'desc': (
            'Skill yang sudah banyak dimiliki dan makin menguat. Ini identitas '
            'kompetensi lulusan: jaga kualitas, kedalaman, dan bukti portofolionya.'
        ),
        'action': 'Rekomendasi: pertahankan sebagai kekuatan inti dan bahan portofolio.',
        'color': '#1d4ed8',
        'fill': '#eff6ff',
    },
    'niche': {
        'label': 'Niche',
        'short': 'NICHE',
        'desc': (
            'Skill langka yang juga melemah. Boleh dipantau, tetapi belum layak '
            'menjadi investasi utama program pelatihan.'
        ),
        'action': 'Rekomendasi: pantau trennya, jangan jadikan prioritas utama.',
        'color': '#64748b',
        'fill': '#f8fafc',
    },
    'mapan': {
        'label': 'Kompetensi Mapan',
        'short': 'KOMPETENSI MAPAN',
        'desc': (
            'Skill yang banyak dimiliki tetapi makin jarang muncul. Risikonya '
            'basi: tetap relevan hanya jika dikombinasikan dengan skill baru.'
        ),
        'action': 'Rekomendasi: jaga relevansi lewat pelatihan lanjutan.',
        'color': '#b45309',
        'fill': '#fefce8',
    },
}

skill_quadrant = [quadrant_of(supply_values[i], momentum_values[i])
                  for i in range(len(hero_skills))]

# Penempatan label agar tidak saling menutupi
label_texts = [s if len(s) <= 20 else s[:19].rstrip() + '…' for s in hero_skills]
label_positions = ['top center'] * len(hero_skills)
placed_boxes = []
char_width = (x_max - 0) / 125.0
gap = y_span * 0.020
box_height = y_span * 0.058


def label_box(x, y, position, text, scale=1.0):
    width = max(len(text), 4) * char_width
    offset = gap * scale
    if position == 'top center':
        return (x - width / 2, x + width / 2, y + offset, y + offset + box_height)
    if position == 'bottom center':
        return (x - width / 2, x + width / 2, y - offset - box_height, y - offset)
    if position == 'top right':
        return (x + offset, x + offset + width, y, y + box_height)
    if position == 'bottom right':
        return (x + offset, x + offset + width, y - box_height, y)
    if position == 'top left':
        return (x - offset - width, x - offset, y, y + box_height)
    if position == 'bottom left':
        return (x - offset - width, x - offset, y - box_height, y)
    if position == 'middle right':
        return (x + offset, x + offset + width, y - box_height / 2, y + box_height / 2)
    return (x - offset - width, x - offset, y - box_height / 2, y + box_height / 2)


def overlap_area(box, others):
    area = 0.0
    for other in others:
        width = min(box[1], other[1]) - max(box[0], other[0])
        height = min(box[3], other[3]) - max(box[2], other[2])
        if width > 0 and height > 0:
            area += width * height
    return area


candidate_positions = [
    'top center', 'bottom center', 'middle right', 'middle left',
    'top right', 'bottom right', 'top left', 'bottom left',
]
marker_half_x = (x_max - 0) * 0.020
marker_half_y = y_span * 0.030
marker_boxes = [
    (x - marker_half_x, x + marker_half_x, y - marker_half_y, y + marker_half_y)
    for x, y in zip(supply_values, momentum_values)
]

for index in np.argsort(supply_values):
    x_value, y_value = supply_values[index], momentum_values[index]
    text = label_texts[index]
    obstacles = placed_boxes + [
        box for other_index, box in enumerate(marker_boxes) if other_index != index
    ]

    best_position, best_box, best_area = 'top center', None, None
    for position in candidate_positions:
        for scale in (1.0, 1.9):
            current = label_box(x_value, y_value, position, text, scale)
            area = overlap_area(current, obstacles)
            if best_area is None or area < best_area:
                best_position, best_box, best_area = position, current, area
            if area == 0:
                break
        if best_area == 0:
            break

    label_positions[index] = best_position
    placed_boxes.append(best_box)

# ==============================================================================
# 7. SUSUN PAYLOAD UNTUK KOMPONEN WEB
# ==============================================================================
priority_list = c6_gap['Skill'].head(3).tolist() if role_ready else []
owned_list = (
    c6_gap.loc[c6_gap['Kepemilikan'] >= 0.5, 'Skill'].head(3).tolist()
    if role_ready else []
)
rising_idx = np.argsort(-momentum_values)[:3]
rising_list = [hero_skills[i] for i in rising_idx]
hub_idx = np.argsort(-np.array([network_strength.get(s, 0) for s in hero_skills]))[:2]
hub_list = [hero_skills[i] for i in hub_idx]


def join_id(items):
    if not items:
        return 'belum tersedia'
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f'{items[0]} dan {items[1]}'
    return ', '.join(items[:-1]) + ', dan ' + items[-1]


def skill_description(idx):
    skill = hero_skills[idx]
    supply = supply_values[idx]
    momentum = momentum_values[idx]
    quad = skill_quadrant[idx]
    role_value = role_required.get(skill) * 100 if skill in role_skill_set else None

    sentences = [
        f"{supply:.0f}% alumni ({int(alumni_values[idx])} orang) mencantumkan "
        f"{skill} di profilnya."
    ]
    if momentum >= 1:
        sentences.append(
            f"Momentumnya naik {momentum:+.0f} poin persentase, artinya makin "
            f"sering muncul pada separuh data terbaru."
        )
    elif momentum <= -1:
        sentences.append(
            f"Momentumnya turun {momentum:.0f} poin persentase, artinya makin "
            f"jarang muncul pada separuh data terbaru."
        )
    else:
        sentences.append("Momentumnya relatif stabil antar periode data.")

    if role_value is not None:
        sentences.append(
            f"Skill ini termasuk kebutuhan {target_role}: {role_value:.0f}% alumni "
            f"di role tersebut memilikinya, dan berada di prioritas "
            f"#{role_rank.get(skill, '-')} untuk dikejar."
        )
    else:
        sentences.append(
            f"Skill ini belum masuk profil {target_role}, sehingga belum menjadi "
            f"kebutuhan utama role tersebut."
        )

    sentences.append(
        f"Terhubung kuat dengan {partners.get(skill, 0)} skill lain "
        f"(skor keterhubungan {network_strength.get(skill, 0)}), sehingga berpotensi "
        f"menjadi pengungkit pembelajaran."
    )
    sentences.append(QUADRANTS[quad]['action'])
    return ' '.join(sentences)


skills_payload = []
for index, skill in enumerate(hero_skills):
    role_value = role_required.get(skill) * 100 if skill in role_skill_set else None
    skills_payload.append({
        'name': skill,
        'type': 'Soft' if skill.lower() in SOFT_SKILLS else 'Hard',
        'supply': round(float(supply_values[index]), 1),
        'momentum': round(float(momentum_values[index]), 1),
        'alumni': int(alumni_values[index]),
        'partners': int(partners.get(skill, 0)),
        'strength': int(network_strength.get(skill, 0)),
        'roleNeed': round(float(role_value), 1) if role_value is not None else None,
        'roleRank': role_rank.get(skill),
        'quadrant': skill_quadrant[index],
        'labelPos': label_positions[index],
        'recentShare': round(float(share_of(count_recent, n_recent, skill)), 1),
        'earlierShare': round(float(share_of(count_earlier, n_earlier, skill)), 1),
        'desc': skill_description(index),
    })

quadrant_payload = {}
for key, info in QUADRANTS.items():
    members = [skill for skill, quad in zip(hero_skills, skill_quadrant) if quad == key]
    member_rows = [
        {
            'name': skill,
            'supply': float(supply_values[hero_skills.index(skill)]),
            'momentum': float(momentum_values[hero_skills.index(skill)]),
            'alumni': int(alumni_values[hero_skills.index(skill)]),
        }
        for skill in members
    ]
    member_rows.sort(key=lambda row: -row['alumni'])
    quadrant_payload[key] = {
        'label': info['label'],
        'desc': info['desc'] + ' ' + info['action'],
        'count': len(members),
        'avgSupply': round(float(np.mean([row['supply'] for row in member_rows])), 1) if member_rows else 0.0,
        'avgMomentum': round(float(np.mean([row['momentum'] for row in member_rows])), 1) if member_rows else 0.0,
        'roleCount': sum(1 for row in member_rows if row['name'] in role_skill_set),
        'members': member_rows[:6],
        'color': info['color'],
        'fill': info['fill'],
    }

if role_ready:
    summary_desc = (
        f"Dari {n_alumni:,} alumni dan {n_unique_skills:,} skill unik, "
        f"{soft_pct:.0f}% skill merupakan soft skills dan {hard_pct:.0f}% hard skills. "
        f"Kesiapan menuju {target_role} baru mencapai {c6_readiness:.0f}%, dengan prioritas "
        f"{join_id(priority_list)}. Skill yang paling menguat: {join_id(rising_list)}. "
        f"Titik pengungkit terbesar: {join_id(hub_list)}."
    )
else:
    summary_desc = (
        f"Dari {n_alumni:,} alumni dan {n_unique_skills:,} skill unik, "
        f"{soft_pct:.0f}% skill merupakan soft skills dan {hard_pct:.0f}% hard skills. "
        f"Role acuan '{target_role}' belum ditemukan pada data, sehingga analisis "
        f"kesiapan role tidak ditampilkan. Skill yang paling menguat: {join_id(rising_list)}. "
        f"Titik pengungkit terbesar: {join_id(hub_list)}."
    )

payload = {
    'meta': {
        'industry': industry_label,
        'role': target_role,
        'roleFound': role_ready,
        'roleHolders': int(len(role_holders)),
        'roleNote': role_note,
        'alumni': int(n_alumni),
        'uniqueSkills': int(n_unique_skills),
        'skillRows': int(n_skill_rows),
        'hardPct': round(float(hard_pct), 1),
        'softPct': round(float(soft_pct), 1),
        'readiness': round(float(c6_readiness), 1),
        'avgSkills': round(float(avg_skills), 1),
        'priority': priority_list,
        'owned': owned_list,
        'summary': summary_desc,
        'quadrants': {key: value['count'] for key, value in quadrant_payload.items()},
    },
    'axis': {
        'xMax': round(float(x_max), 2),
        'yMin': round(float(y_min), 2),
        'yMax': round(float(y_max), 2),
        'xMid': round(float(x_mid), 2),
    },
    'skills': skills_payload,
    'quadrants': quadrant_payload,
}

payload_json = json.dumps(payload, ensure_ascii=False).replace('</', '<\\/')

# ==============================================================================
# 8. TEMPLATE HTML INTERAKTIF (TANPA DEPENDENSI EKSTERNAL)
# ==============================================================================
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Skills Intelligence</title>
<style>
  *{box-sizing:border-box}
  body{margin:0;background:#ffffff}
  .trc{font-family:Arial,Helvetica,sans-serif;color:#0f172a;padding:16px 18px 8px;max-width:1200px;margin:0 auto}
  .trc-head{display:flex;justify-content:space-between;align-items:flex-end;gap:16px;flex-wrap:wrap;margin-bottom:12px}
  .trc-title{font-size:20px;font-weight:700;letter-spacing:-.2px}
  .trc-sub{font-size:11.5px;color:#64748b;margin-top:4px}
  .kpis{display:flex;gap:8px;flex-wrap:wrap}
  .kpi{background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;padding:7px 13px;min-width:92px}
  .kpi b{display:block;font-size:16px;line-height:1.25}
  .kpi span{font-size:9.5px;color:#64748b}
  .trc-body{display:grid;grid-template-columns:1.9fr 1fr;gap:16px;align-items:start}
  .map-card{border:1px solid #e2e8f0;border-radius:14px;padding:8px 10px 4px;background:#fff;position:relative;overflow:hidden}
  .panel{border:1px solid #e2e8f0;border-radius:14px;background:#f8fafc;padding:14px 15px;min-height:560px;display:flex;flex-direction:column}
  .eyebrow{font-size:9.5px;letter-spacing:.09em;color:#94a3b8;font-weight:700}
  h2{font-size:17px;margin:6px 0 2px;line-height:1.25}
  .badges{margin-top:4px}
  .badge{display:inline-block;font-size:9.5px;padding:3px 8px;border-radius:99px;margin:4px 5px 0 0;font-weight:600}
  .badge.hard{background:#e0e7ff;color:#1d4ed8}
  .badge.soft{background:#d1fae5;color:#047857}
  .badge.gold{background:#fef3c7;color:#b45309}
  .badge.gray{background:#e2e8f0;color:#475569}
  .desc{font-size:11.5px;line-height:1.6;color:#334155;margin:10px 0 4px}
  .metrics{display:grid;grid-template-columns:1fr 1fr;gap:7px;margin:10px 0 2px}
  .metric{background:#fff;border:1px solid #e2e8f0;border-radius:9px;padding:7px 9px}
  .metric b{display:block;font-size:15px;line-height:1.3}
  .metric span{font-size:9px;color:#64748b}
  .mini-head{font-size:10.5px;font-weight:700;color:#475569;margin:14px 0 0}
  .mini-note{font-size:9px;color:#94a3b8;margin-top:2px;line-height:1.4}
  .panel-foot{font-size:8.5px;color:#94a3b8;border-top:1px solid #e2e8f0;padding-top:8px;margin-top:auto;line-height:1.55}
  #mini{display:block;width:100%;max-width:320px;height:auto;margin:0 auto}
  .legend{display:flex;gap:13px;font-size:9.5px;color:#64748b;margin:0 0 2px 6px;flex-wrap:wrap}
  .dot{width:8px;height:8px;border-radius:50%;display:inline-block;margin-right:4px}
  .hint{font-size:9px;color:#94a3b8;padding:0 6px 6px}
  .tip{position:absolute;pointer-events:none;background:#0f172a;color:#fff;font-size:10px;line-height:1.5;padding:7px 9px;border-radius:8px;opacity:0;transition:opacity .12s;max-width:230px;z-index:9;box-shadow:0 6px 18px rgba(15,23,42,.25)}
  .tip b{font-size:10.5px}
  #map{display:block;width:100%;height:auto;touch-action:manipulation}
  .node,.quad{cursor:pointer}
  .reset{position:absolute;top:14px;right:16px;font-size:9.5px;color:#64748b;background:#fff;border:1px solid #e2e8f0;border-radius:99px;padding:3px 9px;cursor:pointer}
  .reset:hover{background:#f1f5f9;color:#0f172a}
  @media (max-width:860px){
    .trc-body{grid-template-columns:1fr}
    .panel{min-height:0}
  }
</style>
</head>
<body>
<div class="trc">
  <div class="trc-head">
    <div>
      <div class="trc-title">Skills Intelligence: Peta Peluang Skill</div>
      <div class="trc-sub" id="subtitle"></div>
    </div>
    <div class="kpis" id="kpis"></div>
  </div>

  <div class="trc-body">
    <div class="map-card">
      <div class="legend">
        <span><i class="dot" style="background:#1d4ed8"></i>Hard skill</span>
        <span><i class="dot" style="background:#10b981"></i>Soft skill</span>
        <span><i class="dot" style="background:#fff;border:2px solid #f59e0b"></i>Kebutuhan role</span>
      </div>
      <svg id="map" viewBox="0 0 820 540" role="img" aria-label="Peta peluang skill"></svg>
      <div class="hint">Klik gelembung skill atau label kuadran untuk melihat penjelasannya. Klik area kosong untuk kembali ke ringkasan.</div>
      <div class="tip" id="tip"></div>
      <button class="reset" id="reset">Ringkasan</button>
    </div>

    <aside class="panel">
      <div class="eyebrow" id="p-eyebrow">RINGKASAN</div>
      <h2 id="p-title">Skills Intelligence</h2>
      <div class="badges" id="p-badges"></div>
      <p class="desc" id="p-desc"></p>
      <div class="metrics" id="p-metrics"></div>
      <div class="mini-head" id="p-mini-head"></div>
      <svg id="mini" viewBox="0 0 320 200"></svg>
      <div class="mini-note" id="p-mini-note"></div>
      <div class="panel-foot">Cincin emas = skill yang diminta role target · Ukuran gelembung = jumlah alumni · Warna = hard atau soft skill.</div>
    </aside>
  </div>
</div>

<script>
(function () {
  const DATA = /*__PAYLOAD__*/;
  const META = DATA.meta, AX = DATA.axis, SKILLS = DATA.skills, QUADS = DATA.quadrants;
  const NS = 'http://www.w3.org/2000/svg';

  const W = 820, H = 540, PL = 58, PR = 24, PT = 24, PB = 48;
  const X = v => PL + (v / AX.xMax) * (W - PL - PR);
  const Y = v => H - PB - ((v - AX.yMin) / (AX.yMax - AX.yMin)) * (H - PT - PB);

  const map = document.getElementById('map');
  const tip = document.getElementById('tip');
  const card = map.parentElement;

  function el(name, attrs, parent) {
    const node = document.createElementNS(NS, name);
    for (const key in (attrs || {})) node.setAttribute(key, attrs[key]);
    if (parent) parent.appendChild(node);
    return node;
  }

  function fmtPct(value, digits) {
    return (digits ? value.toFixed(digits) : Math.round(value)) + '%';
  }

  function fmtSigned(value) {
    const rounded = Math.round(value);
    return (rounded > 0 ? '+' : '') + rounded + ' p.p.';
  }

  /* ---------------------------------------------------------------- HEADER */
  document.getElementById('subtitle').textContent =
    'Supply × momentum × keterhubungan × kebutuhan role · Industri: ' + META.industry +
    ' · Role acuan: ' + META.role;

  const kpis = [
    [META.alumni.toLocaleString('id-ID'), 'Alumni'],
    [META.uniqueSkills.toLocaleString('id-ID'), 'Skill unik'],
    [Math.round(META.hardPct) + ' : ' + Math.round(META.softPct), 'Hard : Soft'],
    [META.roleFound ? Math.round(META.readiness) + '%' : '—', 'Readiness ' + META.role],
  ];
  document.getElementById('kpis').innerHTML = kpis.map(function (item) {
    return '<div class="kpi"><b>' + item[0] + '</b><span>' + item[1] + '</span></div>';
  }).join('');

  /* ------------------------------------------------------------------ MAP */
  const quadRects = {};
  const quadGroups = {};
  const quadOrder = [
    ['naik', 0, 0], ['inti', 1, 0], ['niche', 0, 1], ['mapan', 1, 1],
  ];

  quadOrder.forEach(function (entry) {
    const key = entry[0], col = entry[1], row = entry[2];
    const x0 = col === 0 ? 0 : AX.xMid;
    const x1 = col === 0 ? AX.xMid : AX.xMax;
    const y0 = row === 0 ? 0 : AX.yMin;
    const y1 = row === 0 ? AX.yMax : 0;

    const rect = el('rect', {
      x: X(x0), y: Y(y1), width: Math.max(X(x1) - X(x0), 0), height: Math.max(Y(y0) - Y(y1), 0),
      fill: QUADS[key] ? QUADS[key].fill : '#fff', 'fill-opacity': 0.9,
    }, map);
    quadRects[key] = rect;

    const isTop = row === 0;
    const isLeft = col === 0;
    const gx = isLeft ? X(x0) + 12 : X(x1) - 12;
    const gy = isTop ? Y(y1) + 16 : Y(y0) - 16;
    const group = el('g', { class: 'quad' }, map);
    const text = el('text', {
      x: gx, y: gy,
      'text-anchor': isLeft ? 'start' : 'end',
      'font-size': 9.5, 'font-weight': 700, 'letter-spacing': '.06em',
      fill: '#94a3b8',
    }, group);
    text.textContent = (QUADS[key] ? QUADS[key].label.toUpperCase() : key) +
      ' · ' + (QUADS[key] ? QUADS[key].count : 0);
    group.addEventListener('click', function (event) {
      event.stopPropagation();
      selectQuadrant(key);
    });
    group.addEventListener('mouseenter', function () {
      text.setAttribute('fill', '#475569');
    });
    group.addEventListener('mouseleave', function () {
      text.setAttribute('fill', '#94a3b8');
    });
    quadGroups[key] = { group: group, text: text };
  });

  /* Grid & sumbu */
  const gridGroup = el('g', {}, map);
  for (let tick = 0; tick <= 100; tick += 20) {
    if (tick > AX.xMax) break;
    el('line', {
      x1: X(tick), x2: X(tick), y1: PT, y2: H - PB,
      stroke: '#f1f5f9', 'stroke-width': 1,
    }, gridGroup);
    const label = el('text', {
      x: X(tick), y: H - PB + 16, 'text-anchor': 'middle',
      'font-size': 9.5, fill: '#94a3b8',
    }, gridGroup);
    label.textContent = tick + '%';
  }

  const yStep = niceStep((AX.yMax - AX.yMin) / 6);
  const yStart = Math.ceil(AX.yMin / yStep) * yStep;
  for (let value = yStart; value <= AX.yMax; value += yStep) {
    el('line', {
      x1: PL, x2: W - PR, y1: Y(value), y2: Y(value),
      stroke: '#f1f5f9', 'stroke-width': 1,
    }, gridGroup);
    const label = el('text', {
      x: PL - 8, y: Y(value) + 3, 'text-anchor': 'end',
      'font-size': 9.5, fill: '#94a3b8',
    }, gridGroup);
    label.textContent = (value > 0 ? '+' : '') + Math.round(value);
  }

  el('line', {
    x1: PL, x2: W - PR, y1: Y(0), y2: Y(0),
    stroke: '#cbd5e1', 'stroke-width': 1.2, 'stroke-dasharray': '4 4',
  }, map);
  el('line', {
    x1: X(AX.xMid), x2: X(AX.xMid), y1: PT, y2: H - PB,
    stroke: '#cbd5e1', 'stroke-width': 1.2, 'stroke-dasharray': '4 4',
  }, map);

  const axisX = el('text', {
    x: W - PR, y: H - 10, 'text-anchor': 'end', 'font-size': 10, fill: '#64748b',
  }, map);
  axisX.textContent = 'Supply — persentase alumni yang memiliki skill →';

  const axisY = el('text', {
    x: 12, y: PT + 4, 'font-size': 10, fill: '#64748b',
    transform: 'rotate(-90 12 ' + (PT + 4) + ')', 'text-anchor': 'end',
  }, map);
  axisY.textContent = 'Momentum — perubahan pangsa alumni (p.p.) →';

  function niceStep(raw) {
    const power = Math.pow(10, Math.floor(Math.log10(Math.max(raw, 1e-6))));
    const options = [1, 2, 2.5, 5, 10];
    for (let i = 0; i < options.length; i++) {
      if (raw <= options[i] * power) return options[i] * power;
    }
    return 10 * power;
  }

  /* Gelembung skill */
  const maxAlumni = Math.max.apply(null, SKILLS.map(function (skill) { return skill.alumni; }));
  const nodeRefs = [];

  SKILLS.forEach(function (skill, index) {
    const group = el('g', { class: 'node' }, map);
    const radius = 6 + 11 * Math.sqrt(skill.alumni / maxAlumni);
    const hasRole = skill.roleNeed !== null && skill.roleNeed !== undefined;

    const circle = el('circle', {
      cx: X(skill.supply), cy: Y(skill.momentum), r: radius,
      fill: skill.type === 'Soft' ? '#10b981' : '#1d4ed8',
      'fill-opacity': 0.92,
      stroke: hasRole ? '#f59e0b' : '#ffffff',
      'stroke-width': hasRole ? 2.6 : 1.2,
    }, group);

    const text = el('text', labelAttrs(skill, radius), group);
    text.textContent = shortName(skill.name);

    group.addEventListener('click', function (event) {
      event.stopPropagation();
      selectSkill(index);
    });
    group.addEventListener('mouseenter', function () {
      circle.setAttribute('r', radius + 2);
      showTip(skill, event);
    });
    group.addEventListener('mousemove', moveTip);
    group.addEventListener('mouseleave', function () {
      circle.setAttribute('r', radius);
      hideTip();
    });

    nodeRefs.push({ group: group, circle: circle, text: text, radius: radius, hasRole: hasRole });
  });

  function shortName(name) {
    return name.length > 22 ? name.slice(0, 21).replace(/\s+$/, '') + '…' : name;
  }

  function shortLabel(name, limit) {
    return name.length > limit ? name.slice(0, limit - 1).replace(/\s+$/, '') + '…' : name;
  }

  function labelAttrs(skill, radius) {
    const base = {
      'font-size': 9, fill: '#334155', 'pointer-events': 'none',
    };
    switch (skill.labelPos) {
      case 'top center':
        return Object.assign(base, { x: X(skill.supply), y: Y(skill.momentum) - radius - 4, 'text-anchor': 'middle' });
      case 'bottom center':
        return Object.assign(base, { x: X(skill.supply), y: Y(skill.momentum) + radius + 11, 'text-anchor': 'middle' });
      case 'middle right':
        return Object.assign(base, { x: X(skill.supply) + radius + 4, y: Y(skill.momentum) + 3.5, 'text-anchor': 'start' });
      case 'middle left':
        return Object.assign(base, { x: X(skill.supply) - radius - 4, y: Y(skill.momentum) + 3.5, 'text-anchor': 'end' });
      case 'top right':
        return Object.assign(base, { x: X(skill.supply) + radius + 2, y: Y(skill.momentum) - radius, 'text-anchor': 'start' });
      case 'bottom right':
        return Object.assign(base, { x: X(skill.supply) + radius + 2, y: Y(skill.momentum) + radius + 9, 'text-anchor': 'start' });
      case 'top left':
        return Object.assign(base, { x: X(skill.supply) - radius - 2, y: Y(skill.momentum) - radius, 'text-anchor': 'end' });
      default:
        return Object.assign(base, { x: X(skill.supply) - radius - 2, y: Y(skill.momentum) + radius + 9, 'text-anchor': 'end' });
    }
  }

  function showTip(skill, event) {
    let html = '<b>' + skill.name + '</b><br>' +
      'Dimiliki: ' + fmtPct(skill.supply) + ' (' + skill.alumni + ' alumni)<br>' +
      'Momentum: ' + fmtSigned(skill.momentum);
    if (skill.roleNeed !== null && skill.roleNeed !== undefined) {
      html += '<br>Diminta role: ' + fmtPct(skill.roleNeed);
    }
    html += '<br><span style="color:#94a3b8">Klik untuk detail</span>';
    tip.innerHTML = html;
    tip.style.opacity = 1;
    moveTip(event);
  }

  function moveTip(event) {
    const rect = card.getBoundingClientRect();
    let left = event.clientX - rect.left + 14;
    let top = event.clientY - rect.top + 14;
    if (left > rect.width - 235) left = event.clientX - rect.left - 235;
    if (top > rect.height - 90) top = event.clientY - rect.top - 90;
    tip.style.left = left + 'px';
    tip.style.top = top + 'px';
  }

  function hideTip() {
    tip.style.opacity = 0;
  }

  /* -------------------------------------------------------------- PANEL UI */
  let selectedIndex = null;
  let selectedQuadrant = null;

  function setBadges(items) {
    document.getElementById('p-badges').innerHTML = items.map(function (item) {
      return '<span class="badge ' + item.cls + '">' + item.text + '</span>';
    }).join('');
  }

  function setMetrics(items) {
    document.getElementById('p-metrics').innerHTML = items.map(function (item) {
      return '<div class="metric"><b>' + item.value + '</b><span>' + item.label + '</span></div>';
    }).join('');
  }

  function setPanel(config) {
    document.getElementById('p-eyebrow').textContent = config.eyebrow;
    document.getElementById('p-title').textContent = config.title;
    setBadges(config.badges);
    document.getElementById('p-desc').textContent = config.desc;
    setMetrics(config.metrics);
    document.getElementById('p-mini-head').textContent = config.miniHead;
    document.getElementById('p-mini-note').textContent = config.miniNote;
    if (config.donut) {
      renderDonut(config.donut);
    } else {
      renderBars(config.bars);
    }
  }

  function updateMapStyle() {
    nodeRefs.forEach(function (ref, index) {
      const selected = index === selectedIndex;
      const stroke = selected ? '#0f172a' : (ref.hasRole ? '#f59e0b' : '#ffffff');
      ref.circle.setAttribute('stroke', stroke);
      ref.circle.setAttribute('stroke-width', selected ? 3.6 : (ref.hasRole ? 2.6 : 1.2));
      ref.group.setAttribute('opacity', selectedIndex === null || selected ? 1 : 0.55);
      ref.text.setAttribute('opacity', selectedIndex === null || selected ? 1 : 0.55);
    });
    Object.keys(quadRects).forEach(function (key) {
      const active = key === selectedQuadrant;
      quadRects[key].setAttribute('stroke', active ? '#0f172a' : 'none');
      quadRects[key].setAttribute('stroke-width', active ? 1.6 : 0);
      quadRects[key].setAttribute('stroke-dasharray', active ? '5 4' : '0');
      quadRects[key].setAttribute('fill-opacity', selectedQuadrant && !active ? 0.45 : 0.9);
    });
    Object.keys(quadGroups).forEach(function (key) {
      quadGroups[key].text.setAttribute('opacity', selectedIndex === null ? 1 : 0.45);
    });
  }

  /* ----------------------------------------------------------- MINI CHARTS */
  function renderBars(spec) {
    const svg = document.getElementById('mini');
    svg.innerHTML = '';
    const width = 320, rowH = Math.min(30, 176 / Math.max(spec.items.length, 1));
    const labelWidth = 116, barLeft = labelWidth + 8;
    const barMax = width - barLeft - 46;
    const maxValue = spec.max || Math.max.apply(null, spec.items.map(function (item) { return item.value; }).concat([1]));

    spec.items.forEach(function (item, index) {
      const y = index * rowH + 6;
      const barWidth = Math.max((item.value / maxValue) * barMax, 1.5);
      const label = el('text', {
        x: 0, y: y + 13, 'font-size': 9.5, fill: '#475569',
      }, svg);
      label.textContent = shortLabel(item.label, 20);
      el('rect', {
        x: barLeft, y: y + 2, width: barMax, height: 15,
        rx: 4, fill: '#eef2f7',
      }, svg);
      el('rect', {
        x: barLeft, y: y + 2, width: barWidth, height: 15,
        rx: 4, fill: item.color || '#1d4ed8',
      }, svg);
      const outsideX = barLeft + barWidth + 5;
      const placeInside = outsideX > width - 26;
      const value = el('text', {
        x: placeInside ? barLeft + barWidth - 4 : outsideX,
        y: y + 13, 'font-size': 9,
        fill: placeInside ? '#ffffff' : '#64748b',
        'text-anchor': placeInside ? 'end' : 'start',
      }, svg);
      value.textContent = item.suffix ? item.value.toFixed(1) + item.suffix : Math.round(item.value);    });
  }

  function renderDonut(spec) {
    const svg = document.getElementById('mini');
    svg.innerHTML = '';
    const cx = 160, cy = 92, radius = 58, thickness = 17;
    const circumference = 2 * Math.PI * radius;
    const hardLength = circumference * (spec.hard / 100);

    el('circle', {
      cx: cx, cy: cy, r: radius, fill: 'none', stroke: '#10b981',
      'stroke-width': thickness,
    }, svg);
    el('circle', {
      cx: cx, cy: cy, r: radius, fill: 'none', stroke: '#1d4ed8',
      'stroke-width': thickness,
      'stroke-dasharray': hardLength + ' ' + (circumference - hardLength),
      transform: 'rotate(-90 ' + cx + ' ' + cy + ')',
      'stroke-linecap': 'butt',
    }, svg);

    const value = el('text', {
      x: cx, y: cy - 2, 'text-anchor': 'middle', 'font-size': 22,
      'font-weight': 700, fill: '#0f172a',
    }, svg);
    value.textContent = Math.round(spec.hard) + ' : ' + Math.round(spec.soft);

    const caption = el('text', {
      x: cx, y: cy + 18, 'text-anchor': 'middle', 'font-size': 9.5, fill: '#94a3b8',
    }, svg);
    caption.textContent = 'hard : soft';

    const total = el('text', {
      x: cx, y: cy + radius + 26, 'text-anchor': 'middle',
      'font-size': 9.5, fill: '#94a3b8',
    }, svg);
    total.textContent = spec.total.toLocaleString('id-ID') + ' baris skill';
  }

  /* --------------------------------------------------------------- ACTIONS */
  function selectSkill(index) {
    const skill = SKILLS[index];
    selectedIndex = index;
    selectedQuadrant = null;
    updateMapStyle();
    hideTip();

    const badges = [
      { text: skill.type === 'Soft' ? 'Soft skill' : 'Hard skill', cls: skill.type === 'Soft' ? 'soft' : 'hard' },
      { text: QUADS[skill.quadrant].label, cls: 'gray' },
    ];
    if (skill.roleNeed !== null && skill.roleNeed !== undefined) {
      badges.push({ text: 'Kebutuhan ' + META.role, cls: 'gold' });
    }

    const metrics = [
      { value: fmtPct(skill.supply), label: 'Alumni memiliki' },
      { value: fmtSigned(skill.momentum), label: 'Momentum' },
      { value: String(skill.strength.toLocaleString('id-ID')), label: 'Skor keterhubungan' },
      {
        value: skill.roleRank ? '#' + skill.roleRank : '—',
        label: 'Prioritas di role',
      },
    ];

    setPanel({
      eyebrow: 'DETAIL SKILL',
      title: skill.name,
      badges: badges,
      desc: skill.desc,
      metrics: metrics,
      miniHead: 'Kepemilikan vs kebutuhan vs tren',
      miniNote: 'Batang abu-abu = skala 0-100%. Kebutuhan role 0% berarti skill tidak termasuk profil role target.',
      bars: {
        max: 100,
        items: [
          { label: 'Dimiliki alumni', value: skill.supply, color: '#1d4ed8' },
          { label: 'Diminta role', value: skill.roleNeed === null ? 0 : skill.roleNeed, color: '#f59e0b' },
          { label: 'Pangsa data terbaru', value: skill.recentShare, color: '#10b981' },
          { label: 'Pangsa data lama', value: skill.earlierShare, color: '#94a3b8' },
        ],
      },
    });
  }

  function selectQuadrant(key) {
    const info = QUADS[key];
    selectedIndex = null;
    selectedQuadrant = key;
    updateMapStyle();
    hideTip();

    const metrics = [
      { value: String(info.count), label: 'Jumlah skill' },
      { value: fmtPct(info.avgSupply), label: 'Rata-rata supply' },
      { value: fmtSigned(info.avgMomentum), label: 'Rata-rata momentum' },
      { value: String(info.roleCount), label: 'Diminta role target' },
    ];

    setPanel({
      eyebrow: 'KUADRAN',
      title: info.label,
      badges: [
        { text: info.count + ' skill', cls: 'gray' },
        { text: 'Klik gelembung untuk detail', cls: 'gray' },
      ],
      desc: info.desc,
      metrics: metrics,
      miniHead: 'Skill terbesar di kuadran ini',
      miniNote: 'Panjang batang = jumlah alumni yang memiliki skill.',
      bars: {
        items: info.members.map(function (member) {
          return { label: member.name, value: member.alumni, color: info.color };
        }),
      },
    });
  }

  function showSummary() {
    selectedIndex = null;
    selectedQuadrant = null;
    updateMapStyle();
    hideTip();

    setPanel({
      eyebrow: 'RINGKASAN',
      title: 'Skills Intelligence',
      badges: [
        { text: 'Industri: ' + META.industry, cls: 'gray' },
        { text: META.roleFound ? 'Role: ' + META.role : 'Role tidak ditemukan', cls: META.roleFound ? 'gold' : 'gray' },
      ],
      desc: META.summary,
      metrics: [
        { value: META.alumni.toLocaleString('id-ID'), label: 'Alumni dianalisis' },
        { value: META.uniqueSkills.toLocaleString('id-ID'), label: 'Skill unik' },
        { value: Math.round(META.hardPct) + ' : ' + Math.round(META.softPct), label: 'Hard : Soft' },
        { value: META.roleFound ? Math.round(META.readiness) + '%' : '—', label: 'Readiness role' },
      ],
      miniHead: 'Komposisi hard vs soft',
      miniNote: 'Proporsi baris skill, bukan jumlah alumni unik.',
      donut: { hard: META.hardPct, soft: META.softPct, total: META.skillRows },
    });
  }

  map.addEventListener('click', showSummary);
  document.getElementById('reset').addEventListener('click', function (event) {
    event.stopPropagation();
    showSummary();
  });

  showSummary();
})();
</script>
</body>
</html>
"""

html = HTML_TEMPLATE.replace('/*__PAYLOAD__*/', payload_json)

# ==============================================================================
# 9. TAMPILKAN & SIMPAN VERSI MANDIRI UNTUK WEB
# ==============================================================================
try:
    from IPython.display import HTML, display
    display(HTML(html))
except Exception as exc:
    print(f"Pratinjau inline tidak tersedia ({exc}). Buka file HTML hasil ekspor.")

if export_html_path:
    Path(export_html_path).write_text(html, encoding='utf-8')

# ==============================================================================
# 10. RINGKASAN TEKS
# ==============================================================================
print("=" * 62)
print("SKILLS INTELLIGENCE — DIAGRAM INTERAKTIF")
print("=" * 62)
print(f"Industri              : {industry_label}")
print(f"Alumni dianalisis     : {n_alumni:,}")
print(f"Skill di diagram      : {len(skills_payload)}")
if role_ready:
    print(f"Role acuan            : {target_role} ({len(role_holders)} alumni, {role_note})")
    print(f"Readiness             : {c6_readiness:.1f}%")
    print(f"Prioritas utama       : {join_id(priority_list)}")
else:
    print(f"Role acuan            : {target_role} (tidak ditemukan)")
print(f"Momentum teratas      : {join_id(rising_list)}")
print(f"Titik pengungkit      : {join_id(hub_list)}")
print(f"Kuadran               : " + ' · '.join(
    f"{info['label']} {info['count']}" for info in quadrant_payload.values()
))
if export_html_path:
    print(f"File HTML mandiri     : {Path(export_html_path).resolve()}")
print("=" * 62)

SKILLS INTELLIGENCE — DIAGRAM INTERAKTIF
Industri              : Semua
Alumni dianalisis     : 394
Skill di diagram      : 20
Role acuan            : Data Analyst (3 alumni, riwayat jabatan)
Readiness             : 9.7%
Prioritas utama       : Python, Finance, dan Analytical Skills
Momentum teratas      : Customer Service, Problem Solving, dan Administration
Titik pengungkit      : Communication dan Leadership
Kuadran               : Naik Daun 3 · Kekuatan Inti 3 · Niche 7 · Kompetensi Mapan 7
File HTML mandiri     : D:\OJT BLKPP\TraceraVarguard\ColabSkills2\tracera\skills_intelligence.html
